# Chapter 6 — Lenses

Companion tutorial for **[Foundations of Computer Vision](https://visionbook.mit.edu/lenses.html)** by Torralba, Isola, and Freeman (MIT Press, 2024).

This is the lens chapter of Part II (Image Formation), and it answers a question pinholes can't: how do you let in more light without sacrificing sharpness? The answer starts with Snell's law. The first section derives the **lensmaker's formula**, which turns Snell's law plus a small-angle approximation into one equation,

$$\frac{1}{a} + \frac{1}{b} = \frac{1}{f},$$

relating object distance $a$, image distance $b$, and focal length $f$.

Each figure is a runnable construction: change a refractive index, a radius of curvature, or an object distance, and the diagram redraws from the new physics. The chapter's diagrammatic figures are reproduced directly; where the chapter shows a photograph, a synthetic version computed from the same physics stands in.

In [ ]:
from pathlib import Path

import torch
import kornia
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle, Circle, FancyArrowPatch
from matplotlib.collections import LineCollection

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(0)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": False,
})

# Locate the chapter's assets and image output regardless of the working directory.
_CHAPTER_DIR = Path("notebooks/CV/mit-foundations/chapter-6-lenses")
BASE_DIR = _CHAPTER_DIR if _CHAPTER_DIR.is_dir() else Path(".")
ASSETS_DIR = BASE_DIR / "assets"
IMAGES_DIR = BASE_DIR / "images"
IMAGES_DIR.mkdir(exist_ok=True)


In [ ]:
# Section 6.2 — physics primitives.
# Snell's law and its paraxial small-angle form, the surface-tilt angle, the
# lensmaker's focal length and conjugate image distance, and the four-angle
# bookkeeping that the thin-lens derivation composes from them. (Book §6.2.)


def snells_law(n1, n2, theta1):
    """Refraction angle θ₂ at a flat interface (n₁ sin θ₁ = n₂ sin θ₂).

    Angles in radians, measured from the surface normal. Returns NaN past
    the critical angle (total internal reflection).
    """
    n1 = torch.as_tensor(n1, dtype=torch.float32)
    n2 = torch.as_tensor(n2, dtype=torch.float32)
    theta1 = torch.as_tensor(theta1, dtype=torch.float32)
    return torch.asin((n1 / n2) * torch.sin(theta1))


def paraxial_snell(n1, n2, theta1):
    """Paraxial Snell's law: n1·θ1 = n2·θ2, the small-angle form of snells_law.

    Differs from snells_law by sin(θ) ≈ θ. Used throughout §6.2's lensmaker
    derivation where the four ray angles all sit in the paraxial regime.
    Angles in radians, measured from the surface normal.
    """
    n1 = torch.as_tensor(n1, dtype=torch.float32)
    n2 = torch.as_tensor(n2, dtype=torch.float32)
    theta1 = torch.as_tensor(theta1, dtype=torch.float32)
    return (n1 / n2) * theta1


def surface_angle(c, R):
    """Tilt angle θ_S of a spherical surface at height c, radius R.

    Paraxial form: θ_S ≈ c/R. (Book §6.2, Figure 6.5.)
    """
    c = torch.as_tensor(c, dtype=torch.float32)
    R = torch.as_tensor(R, dtype=torch.float32)
    return c / R


def focal_length(n, R1, R2):
    """Focal length of a thin lens via the lensmaker's equation.

    1/f = (n-1) * (1/R1 + 1/R2).  (Book §6.2, Eq. derived from Table 6.1.)
    Sign convention: both R1 and R2 are positive magnitudes. A biconvex lens
    has R1 > 0 and R2 > 0. (Book §6.2 — note this differs from Hecht 2016,
    which uses signed radii.)
    """
    n = torch.as_tensor(n, dtype=torch.float32)
    R1 = torch.as_tensor(R1, dtype=torch.float32)
    R2 = torch.as_tensor(R2, dtype=torch.float32)
    return 1.0 / ((n - 1.0) * (1.0 / R1 + 1.0 / R2))


def image_distance(a, f):
    """Image distance b from object distance a and focal length f.

    1/a + 1/b = 1/f  =>  b = a*f / (a - f).  (Book §6.2.)
    Returns inf when a == f (object at the focal point → image at infinity).
    """
    a = torch.as_tensor(a, dtype=torch.float32)
    f = torch.as_tensor(f, dtype=torch.float32)
    return a * f / (a - f)


def lensmaker_angle_bookkeeping(n, R1, R2, c, theta1_deg):
    """Compute the four paraxial ray angles and two surface tilts through a thin lens.

    Applies paraxial Snell's law at each surface using the helpers defined
    above. Returns (theta1_axis, theta2_axis, theta4_axis, thetaS1, thetaS2),
    all in radians measured from the optical axis (rays) or from horizontal
    (surface tilts). (Book §6.2, Table 6.1.)
    """
    theta1_axis = torch.deg2rad(theta1_deg)

    # Surface tilts at the ray's hit height (both radii now positive magnitudes
    # per the book's convention).
    thetaS1 = surface_angle(c, R1)
    thetaS2 = surface_angle(c, R2)

    # Refract entering the glass at the front surface.
    theta1_normal = theta1_axis + thetaS1
    theta2_normal = paraxial_snell(1.0, n, theta1_normal)
    theta2_axis = theta2_normal - thetaS1

    # Refract exiting the glass at the back surface.
    theta3_normal = thetaS2 - theta2_axis
    theta4_normal = paraxial_snell(n, 1.0, theta3_normal)
    theta4_axis = thetaS2 - theta4_normal

    return theta1_axis, theta2_axis, theta4_axis, thetaS1, thetaS2


# Sanity checks — one per function.
# Snell's law: 30° in air → ≈19.47° in glass (n=1.5).
print(f"{torch.rad2deg(snells_law(1.0, 1.5, torch.deg2rad(torch.tensor(30.0)))).item():.2f}°")
# Paraxial Snell: 5° in air → ≈3.33° in glass (n=1.5), close to exact 3.33°.
print(f"{torch.rad2deg(paraxial_snell(1.0, 1.5, torch.deg2rad(torch.tensor(5.0)))).item():.2f}°")
# Surface tilt: radius 50 mm, ray hit at height 5 mm → about 0.1 rad ≈ 5.73°.
print(f"{torch.rad2deg(surface_angle(5.0, 50.0)).item():.2f}°")
# Biconvex lens, n=1.5, R1=R2=50 mm → f = 50 mm. Object at 200 mm → b ≈ 67 mm.
f = focal_length(1.5, 50.0, 50.0)
b = image_distance(200.0, f)
print(f"f = {f.item():.2f} mm, b(a=200) = {b.item():.2f} mm")

In [211]:
# Shared helpers — drawing primitives reused across Chapter 6 figures.


# Shared color palette — book-style figures use a consistent set of colors
# across panels so the eye reads "red = distance bracket" and "blue = ray"
# without relearning each figure. Per-cell overrides are acceptable when a
# figure genuinely needs something different (e.g. the cyan lens-surface
# lines in Fig 6.8 only appear there).
COLORS = {
    "dark":   "#111111",   # axes, ray lines, primary text
    "gray":   "#555555",   # secondary text, dashed reference lines
    "blue":   "#1f6fe0",   # rays
    "red":    "#c0392b",   # distance brackets (a, b, f, c, d)
    "green":  "#3a8b3a",   # angles
    "cyan":   "#2997ff",   # lens surfaces in §6.3 perspective-projection figures
    "panel":  "#5f7695",   # panel labels "(a)", "(b)" in multi-panel figures
    "normal": "#45b8c8",   # surface-normal lines and right-angle marks in 6.4(b)
}


def draw_angle(ax, center, angle_a, angle_b, radius, text=None, text_radius=None, text_shift=None, fs=9.8, color="0.25", lw=0.85, z=7):
    """Draw one angle arc between two tensor-defined directions."""
    # Sweep the smaller signed arc from angle_a toward angle_b.
    delta = torch.atan2(torch.sin(angle_b - angle_a), torch.cos(angle_b - angle_a))
    a0, a1 = angle_a, angle_a + delta
    # Sample the arc directly as a tensor span — no .item() round-trip into linspace.
    t = a0 + (a1 - a0) * torch.linspace(0.0, 1.0, 80, dtype=torch.float32)

    x = center[0] + radius * torch.cos(t)
    y = center[1] + radius * torch.sin(t)
    ax.plot([v.item() for v in x], [v.item() for v in y], color=color, lw=lw, zorder=z)

    if text is not None:
        if text_radius is None:
            text_radius = radius + torch.tensor(0.20, dtype=torch.float32)
        if text_shift is None:
            text_shift = torch.tensor([0.0, 0.0], dtype=torch.float32)

        mid_angle = 0.5 * (a0 + a1)
        text_point = center + text_radius * torch.stack([torch.cos(mid_angle), torch.sin(mid_angle)]) + text_shift
        ax.text(text_point[0].item(), text_point[1].item(), text, fontsize=fs, color=color, ha="center", va="center")


def lens_curves(gap, bulge, half_h, samples=320):
    """Compute the two curved sides of a biconvex lens."""
    y = torch.linspace(-half_h, half_h, samples)
    y_unit = y / half_h
    profile = torch.sqrt(torch.clamp(1.0 - y_unit**2, min=0.0))
    left_x = -gap - bulge * profile
    right_x = gap + bulge * profile
    return left_x, right_x, y


def surface_x_at_y(y, gap, bulge, half_h, side):
    """Compute where a curved lens surface sits at a given ray height."""
    y_unit = y / half_h
    profile = torch.sqrt(torch.clamp(1.0 - y_unit**2, min=0.0))
    if side == "left":
        return -gap - bulge * profile
    return gap + bulge * profile


def draw_biconvex_lens(ax, left_x, right_x, y, face="#dbe7f5", lw=1.45, alpha=0.95):
    """Draw a filled biconvex lens body from tensor-defined surface curves."""
    boundary = [(left_x[i].item(), y[i].item()) for i in range(y.numel())]
    boundary += [(right_x[i].item(), y[i].item()) for i in range(y.numel() - 1, -1, -1)]

    ax.add_patch(
        Polygon(
            boundary,
            closed=True,
            facecolor=face,
            edgecolor="none",
            alpha=alpha,
            zorder=1,
        )
    )
    ax.plot([v.item() for v in left_x], [v.item() for v in y], color="black", lw=lw, zorder=3)
    ax.plot([v.item() for v in right_x], [v.item() for v in y], color="black", lw=lw, zorder=3)


def solve_surface_hit_from_ray(front_hit, theta_axis, gap, bulge, half_h):
    """Find where the in-glass ray intersects the back curved surface."""
    slope = torch.tan(theta_axis)
    y_candidates = torch.linspace(-0.95 * half_h, 0.95 * half_h, 2000)
    x_candidates = surface_x_at_y(y_candidates, gap, bulge, half_h, side="right")
    y_on_ray = front_hit[1] + slope * (x_candidates - front_hit[0])
    idx = torch.argmin(torch.abs(y_candidates - y_on_ray))
    return torch.stack([x_candidates[idx], y_candidates[idx]])

## 6.1 — Introduction

A pinhole camera works: light from a scene passes through a small opening and forms an image on a sensor on the other side. But pinholes force a trade-off. Shrink the aperture and each scene point maps to a tight spot — the image is sharp, but so little light gets through that the image is dim. Open the aperture and more light arrives, but each scene point now spreads across many sensor pixels — the image is bright but blurry. There's no single pinhole size that's both sharp and bright.

A lens breaks the trade-off. It gathers the wide cone of light a large aperture admits and *refocuses* it back to a single point on the sensor — bright like the wide pinhole, sharp like the narrow one. The rest of the chapter is the geometry of how a lens does that.

**Figure 6.1 — Brightness/sharpness trade-offs in image formation.** Three setups, same scene at the top, same sensor at the bottom. **(a) Small pinhole:** sharp image, very dim — little light reaches the sensor. **(b) Large pinhole:** bright image, very blurry — each scene point spreads to a wide disk at the sensor. **(c) Lens:** bright AND sharp — the lens collects a wide cone of light from each scene point and refocuses it to one sensor point. (Book §6.1, Figure 6.1.)

In [ ]:
# Figure 6.1 — Brightness/sharpness trade-offs in image formation (book §6.1).
#
# Three columns compare a small pinhole, a large pinhole, and a lens. The
# scene image is the same in all three cases; changing the opening controls
# how much light reaches the sensor and how tightly one scene point maps to
# one sensor location.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
small_ap = torch.tensor(0.08, dtype=torch.float32)
large_ap = torch.tensor(0.42, dtype=torch.float32)
lens_ap = torch.tensor(0.26, dtype=torch.float32)
scene_to_ap = torch.tensor(1.45, dtype=torch.float32)
ap_to_sensor = torch.tensor(1.18, dtype=torch.float32)
dark_gain = torch.tensor(0.34, dtype=torch.float32)
blur_kernel_ratio = torch.tensor(13.0, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
scene_path = ASSETS_DIR / "dog_scene.png"
assert scene_path.exists(), f"missing scene asset at {scene_path}"
x_cols = torch.tensor([1.8, 5.2, 8.6], dtype=torch.float32)
scene_w = torch.tensor(1.45, dtype=torch.float32)
scene_h = torch.tensor(1.45, dtype=torch.float32)
img_w = torch.tensor(1.45, dtype=torch.float32)
img_h = torch.tensor(1.45, dtype=torch.float32)
y_scene = torch.tensor(7.65, dtype=torch.float32)
y_ap = torch.tensor(5.35, dtype=torch.float32)
y_sensor = torch.tensor(4.05, dtype=torch.float32)
y_image = torch.tensor(2.35, dtype=torch.float32)
sensor_w = torch.tensor(2.25, dtype=torch.float32)
sensor_h = torch.tensor(0.12, dtype=torch.float32)
bar_w = torch.tensor(2.05, dtype=torch.float32)
bar_lw = 8.5
lens_h = torch.tensor(0.18, dtype=torch.float32)
lens_w = torch.tensor(0.16, dtype=torch.float32)
ray_orange = "#ff5a1f"
ray_red = "#ff2f2f"
dark = COLORS["dark"]
gray = COLORS["gray"]

# --- Derived quantities ---
scene = torch.tensor(plt.imread(scene_path)[..., :3], dtype=torch.float32)
if scene.max() > 1.0:
    scene = scene / 255.0

# Small pinhole — sharp but dim (little light reaches the sensor).
img_dark = torch.clamp(scene * dark_gain, 0.0, 1.0)

# Large pinhole — bright but blurry (each scene point spreads to a disk at the sensor).
# Kernel size scales with aperture ratio: larger aperture, larger blur.
blur_k = max(5, int((blur_kernel_ratio * (large_ap / small_ap)).round().item()))
blur_k += 1 - blur_k % 2  # force odd
scene_chw = scene.permute(2, 0, 1).unsqueeze(0)
img_blur = kornia.filters.box_blur(scene_chw, (blur_k, blur_k))
img_blur = img_blur.squeeze(0).permute(1, 2, 0).clamp(0.0, 1.0)

panel_imgs = [img_dark, img_blur, scene]

src_x = x_cols - torch.tensor([0.18, 0.36, 0.22], dtype=torch.float32)
src_y = torch.full_like(x_cols, y_scene - 0.55)
hit_x = x_cols + torch.tensor([0.12, 0.04, 0.36], dtype=torch.float32)
hit_y = torch.full_like(x_cols, y_sensor)

lens_y = torch.linspace(-lens_ap, lens_ap, 120, dtype=torch.float32)
lens_x = lens_w * torch.sqrt(torch.clamp(1.0 - (lens_y / lens_ap) ** 2, min=0.0))

# --- Drawing ---
fig, ax = plt.subplots(figsize=(12.8, 4.8))

# Scene (top) and image-at-sensor (bottom) panels for all three columns.
for xc, im in zip(x_cols, panel_imgs):
    for y_center, image in [(y_image, im), (y_scene, scene)]:
        ax.imshow(
            image,
            extent=[
                (xc - scene_w / 2).item(), (xc + scene_w / 2).item(),
                (y_center - img_h / 2).item(), (y_center + img_h / 2).item(),
            ],
            zorder=0,
        )

# Aperture row — pinhole bars for columns 0 and 1, lens body for column 2.
for i, xc in enumerate(x_cols.tolist()):
    left = xc - bar_w.item() / 2
    right = xc + bar_w.item() / 2
    opening = small_ap.item() if i == 0 else large_ap.item() if i == 1 else lens_w.item()
    ax.plot([left, xc - opening], [y_ap.item(), y_ap.item()], color=dark, lw=bar_lw, solid_capstyle="butt", zorder=3)
    ax.plot([xc + opening, right], [y_ap.item(), y_ap.item()], color=dark, lw=bar_lw, solid_capstyle="butt", zorder=3)

# Lens body sits in the opening of the third column.
lens_poly = torch.cat(
    [
        torch.stack([x_cols[2] - lens_x, y_ap + lens_y], dim=1),
        torch.stack([x_cols[2] + lens_x.flip(0), (y_ap + lens_y).flip(0)], dim=1),
    ],
    dim=0,
)
ax.add_patch(Polygon(lens_poly.tolist(), closed=True, facecolor="white", edgecolor=dark, lw=1.0, zorder=4))

# Camera sensor — hatched strip below the aperture, one per column.
for xc in x_cols.tolist():
    x0 = xc - sensor_w.item() / 2
    x1 = xc + sensor_w.item() / 2
    for y in [y_sensor, y_sensor - sensor_h]:
        ax.plot([x0, x1], [y.item(), y.item()], color=dark, lw=1.2, zorder=3)
    for t in torch.linspace(x0, x1, 17):
        ax.plot([t.item(), t.item()], [(y_sensor - sensor_h).item(), y_sensor.item()], color=dark, lw=0.8, zorder=3)

# Ray cones — one scene point through each aperture down to the sensor.
# Small pinhole: narrow cone, narrow spot. Two overlapping triangles for visual depth.
xc = x_cols[0]
for x_offset, color, alpha in [(-small_ap.item() / 2, ray_orange, 0.95), (+small_ap.item() / 2, ray_red, 0.75)]:
    ax.fill(
        [src_x[0].item(), xc.item() + x_offset, hit_x[0].item()],
        [src_y[0].item(), y_ap.item(), hit_y[0].item()],
        color=color, alpha=alpha, zorder=2,
    )

# Large pinhole: wide quadrilateral fan.
xc = x_cols[1]
ax.fill(
    [src_x[1].item(), xc.item() - large_ap.item() / 2, hit_x[1].item(), xc.item() + large_ap.item() / 2],
    [src_y[1].item(), y_ap.item(), hit_y[1].item(), y_ap.item()],
    color=ray_orange, alpha=0.95, zorder=2,
)

# Lens: wide cone refocused to a single point. Same two-triangle pattern as small pinhole.
xc = x_cols[2]
for x_offset, color, alpha in [(-lens_w.item(), ray_orange, 0.90), (+lens_w.item(), ray_red, 0.70)]:
    ax.fill(
        [src_x[2].item(), xc.item() + x_offset, hit_x[2].item()],
        [src_y[2].item(), y_ap.item(), hit_y[2].item()],
        color=color, alpha=alpha, zorder=2,
    )

# Row labels on the right side of the figure.
for y, txt in [
    (y_scene, "Scene"),
    (y_ap, "Aperture"),
    (y_sensor - sensor_h / 2, "Camera\nsensor"),
    (y_image, "Image at\nsensor"),
]:
    ax.text(10.55, y.item(), txt, fontsize=12, color=dark, ha="left", va="center")

# Column labels below the panels.
for xc, lab in zip(x_cols, ["small pinhole", "large pinhole", "lens"]):
    ax.text(xc.item(), 0.48, lab, fontsize=12, color=gray, ha="center", va="center")

ax.set_xlim(0.35, 12.15)
ax.set_ylim(0.15, 8.55)
ax.axis("off")

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_01.png", dpi=150, bbox_inches="tight")
plt.show()

The next section derives the geometry behind panel (c): how exactly does a lens refocus a wide cone of light from each scene point back to a single sensor point? The answer is Snell's law applied twice (once at each glass surface), with a small-angle approximation that linearizes the algebra. That's the **lensmaker's formula**, which §6.2 builds next.

## 6.2 — Lensmaker's Formula

A lens refracts light at each of its two surfaces. The lensmaker's formula compresses that two-refraction process into one equation,

$$\frac{1}{a} + \frac{1}{b} = \frac{1}{f},$$

relating object distance $a$, image distance $b$, and focal length $f$. It follows from applying Snell's law twice with the small-angle approximation $\sin\theta \approx \theta$ — the *paraxial* regime where the algebra stays linear. (Book §6.2.)

For small angles in radians, $\sin\theta \approx \theta$, and Snell's law at a glass-air interface ($n_1 = 1$, $n_2 = n$) becomes the linear $\theta_1 = n\theta_2$. The book uses this paraxial form throughout the rest of the section.

**Figure 6.3(a) — Snell's law at a flat interface.** A ray crosses from a medium with refractive index $n_1$ into a denser medium with $n_2 > n_1$ and bends toward the normal. The angles $\theta_1, \theta_2$ are measured from the normal, and $n_1 \sin\theta_1 = n_2 \sin\theta_2$ relates them. The book's panel (b), a photograph of a straw refracting in a glass of water, is the same physics in the physical world; that photo is not reproduced here. (Book §6.2, Figure 6.3.)

In [ ]:
# Figure 6.3 (a) — Snell's law at a flat interface (book §6.2).
#
# This cell draws one incident ray crossing a flat interface between two media.
# The tunable parameters below set the two refractive indices and the incident
# angle; rerunning the cell recomputes the refracted angle from Snell's law and
# redraws the ray, angle markers, and labels from those physical values.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. If n1 * sin(theta1) exceeds n2, total internal reflection
# occurs and this transmitted-ray drawing is no longer the correct physical case.
n1 = torch.tensor(1.0, dtype=torch.float32)
n2 = torch.tensor(1.5, dtype=torch.float32)
theta1_deg = torch.tensor(35.0, dtype=torch.float32)

ray_len = torch.tensor(1.45, dtype=torch.float32)
angle_radius = torch.tensor(0.42, dtype=torch.float32)
label_radius = torch.tensor(0.64, dtype=torch.float32)
interface_half_width = torch.tensor(1.70, dtype=torch.float32)
normal_half_height = torch.tensor(1.55, dtype=torch.float32)

# --- Derived quantities ---
zero = torch.tensor(0.0, dtype=torch.float32)
pi = torch.tensor(torch.pi, dtype=torch.float32)

theta1 = torch.deg2rad(theta1_deg)
theta2 = snells_law(n1, n2, theta1)

origin = torch.stack([zero, zero])

# Directions are measured from the surface normal. The incident ray travels
# down and right toward the interface; the refracted ray leaves down and right
# with the angle recomputed by Snell's law.
incident_dir = torch.stack([torch.sin(theta1), -torch.cos(theta1)])
refracted_dir = torch.stack([torch.sin(theta2), -torch.cos(theta2)])

source = -ray_len * incident_dir
incoming_pre_hit = -torch.tensor(0.12, dtype=torch.float32) * incident_dir
refracted_tip = ray_len * refracted_dir

# Reference directions for the angle markers.
normal_up = 0.5 * pi
normal_down = -0.5 * pi
incident_from_origin = normal_up + theta1
refracted_from_origin = normal_down + theta2

dark, gray = COLORS["dark"], COLORS["gray"]
# Per-figure tints: kept local because they're specific to this Snell-law figure's
# two-medium fill and the dark-blue label color isn't reused elsewhere.
blue_tint_top = "#eef4fb"
blue_tint_bottom = "#d4e1f0"
label_blue = "#1f3a5f"

# --- Drawing ---
fig, ax = plt.subplots(figsize=(6.2, 5.0))

# Fill the two media on either side of the flat interface.
ax.axhspan(0.0, interface_half_width.item(), color=blue_tint_top, zorder=0)
ax.axhspan(-interface_half_width.item(), 0.0, color=blue_tint_bottom, zorder=0)

# Draw the interface and the normal through the incidence point.
ax.plot(
    [-interface_half_width.item(), interface_half_width.item()],
    [0.0, 0.0],
    color="0.20",
    lw=1.3,
    zorder=2,
)
ax.plot(
    [0.0, 0.0],
    [-normal_half_height.item(), normal_half_height.item()],
    color=gray,
    lw=1.0,
    ls="--",
    zorder=2,
)

# Draw the incident ray as a line ending at the interface.
ax.plot(
    [source[0].item(), origin[0].item()],
    [source[1].item(), origin[1].item()],
    color=dark,
    lw=1.4,
    zorder=3,
)

# Add a small direction marker on the incident ray.
incoming_arrow_start = source + torch.tensor(0.42, dtype=torch.float32) * incident_dir
incoming_arrow_end = source + torch.tensor(0.72, dtype=torch.float32) * incident_dir
ax.add_patch(
    FancyArrowPatch(
        (incoming_arrow_start[0].item(), incoming_arrow_start[1].item()),
        (incoming_arrow_end[0].item(), incoming_arrow_end[1].item()),
        arrowstyle="-|>",
        mutation_scale=13,
        lw=1.4,
        color=dark,
        zorder=4,
    )
)

# Draw the refracted ray leaving the interface.
ax.add_patch(
    FancyArrowPatch(
        (origin[0].item(), origin[1].item()),
        (refracted_tip[0].item(), refracted_tip[1].item()),
        arrowstyle="-|>",
        mutation_scale=13,
        lw=1.4,
        color=dark,
        shrinkA=0,
        shrinkB=0,
        zorder=3,
    )
)

# Mark the two measured angles from the normal.
draw_angle(
    ax,
    origin,
    normal_up,
    incident_from_origin,
    angle_radius,
    r"$\theta_1$",
    text_radius=label_radius,
    text_shift=torch.tensor([0.02, 0.02], dtype=torch.float32),
    fs=13,
    color=dark,
    lw=1.0,
    z=5,
)
draw_angle(
    ax,
    origin,
    normal_down,
    refracted_from_origin,
    angle_radius,
    r"$\theta_2$",
    text_radius=label_radius,
    text_shift=torch.tensor([0.02, -0.02], dtype=torch.float32),
    fs=13,
    color=dark,
    lw=1.0,
    z=5,
)

# Label the two media using the current refractive-index values.
ax.text(
    1.02,
    1.25,
    rf"$n_1 = {n1.item():.1f}$",
    ha="center",
    va="center",
    fontsize=14,
    color=label_blue,
)
ax.text(
    -1.02,
    -1.25,
    rf"$n_2 = {n2.item():.1f}$",
    ha="center",
    va="center",
    fontsize=14,
    color=label_blue,
)

# Place Snell's law in the open lower-right part of the denser medium.
ax.text(
    1.00,
    -1.42,
    r"$n_1 \sin\theta_1 = n_2 \sin\theta_2$",
    ha="center",
    va="center",
    fontsize=11,
    color=label_blue,
)

ax.set_xlim(-1.65, 1.65)
ax.set_ylim(-1.65, 1.65)
ax.set_aspect("equal", adjustable="box")
ax.axis("off")

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_03a.png", dpi=150, bbox_inches="tight")
plt.show()

**Figure 6.4(a) — Thin-lens geometry.** A point on the optical axis at distance $a$ from the lens emits rays in many directions. Those rays pass through the lens at various heights up to $c$ and converge to a single image point at distance $b$ on the other side. The angle the upper extreme ray makes with the optical axis is $\theta_1$ on the object side and $\theta_4$ on the image side. The lensmaker's formula derived in this section is the statement that for a thin lens, this convergence happens at the same $b$ for every ray emitted from the same object point. (Book §6.2, Figure 6.4(a).)

In [ ]:
# Figure 6.4 (a) — Thin-lens geometry (book §6.2).
#
# This cell draws the single-ray thin-lens construction: one ray leaves the
# object, reaches the lens at height c, and reconverges to the image. The
# physical inputs are n, R1, R2, a, and c; f, b, theta_1, and theta_4 are
# computed from them, so rerunning after a change updates the whole figure.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
n = torch.tensor(1.5, dtype=torch.float32)
R1 = torch.tensor(1.4, dtype=torch.float32)
R2 = torch.tensor(1.4, dtype=torch.float32)
a = torch.tensor(4.0, dtype=torch.float32)
c = torch.tensor(0.85, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
lens_gap = torch.tensor(0.0, dtype=torch.float32)
lens_bulge = torch.tensor(0.18, dtype=torch.float32)
lens_half_h = torch.tensor(1.15, dtype=torch.float32)
arc_r = torch.tensor(0.44, dtype=torch.float32)
br_y = torch.tensor(-0.42, dtype=torch.float32)
tick = torch.tensor(0.055, dtype=torch.float32)

# --- Derived quantities ---
z = torch.tensor(0.0, dtype=torch.float32)
f = focal_length(n, R1, R2)
b = image_distance(a, f)
theta1 = torch.atan(c / a)
theta4 = torch.atan(c / b)

obj = torch.stack([-a, z])
hit = torch.stack([z, c])
img = torch.stack([b, z])

front_vertex = torch.stack([-lens_gap, z])
back_vertex = torch.stack([lens_gap, z])

left_x, right_x, lens_y = lens_curves(lens_gap, lens_bulge, lens_half_h)

t = torch.linspace(0.0, 1.0, 60, dtype=torch.float32)
left_arc = t * theta1
right_arc = torch.pi - t * theta4

incoming_u = torch.stack([torch.cos(theta1), torch.sin(theta1)])
outgoing_u = torch.stack([torch.cos(theta4), -torch.sin(theta4)])

# --- Drawing ---
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot([(-a - 0.45).item(), (b + 0.45).item()], [0.0, 0.0], color=COLORS["dark"], lw=1.0, zorder=0)
ax.text(-1.25, 0.16, "optical axis", fontsize=9, color=COLORS["gray"], ha="center", va="bottom")

# Draw the lens first so the ray can sit visibly on top of it at the bend.
draw_biconvex_lens(ax, left_x, right_x, lens_y, face="white", lw=1.2, alpha=1.0)
ax.text(0.0, -1.33, "thin lens", fontsize=9, color=COLORS["gray"], ha="center", va="top")

# Draw one continuous bent ray, with the bend visible at the lens.
ax.plot([obj[0].item(), hit[0].item()], [obj[1].item(), hit[1].item()], color=COLORS["blue"], lw=1.3, zorder=5)
ax.plot([hit[0].item(), img[0].item()], [hit[1].item(), img[1].item()], color=COLORS["blue"], lw=1.3, zorder=5)

# Direction markers sit on the ray, not behind the lens.
p_in0 = obj + torch.tensor(1.10, dtype=torch.float32) * incoming_u
p_in1 = obj + torch.tensor(1.45, dtype=torch.float32) * incoming_u
ax.add_patch(
    FancyArrowPatch(
        (p_in0[0].item(), p_in0[1].item()),
        (p_in1[0].item(), p_in1[1].item()),
        arrowstyle="-|>",
        mutation_scale=9,
        lw=1.0,
        color=COLORS["blue"],
        zorder=6,
    )
)

p_out0 = hit + torch.tensor(0.85, dtype=torch.float32) * outgoing_u
p_out1 = hit + torch.tensor(1.20, dtype=torch.float32) * outgoing_u
ax.add_patch(
    FancyArrowPatch(
        (p_out0[0].item(), p_out0[1].item()),
        (p_out1[0].item(), p_out1[1].item()),
        arrowstyle="-|>",
        mutation_scale=9,
        lw=1.0,
        color=COLORS["blue"],
        zorder=6,
    )
)

ax.scatter([obj[0].item(), img[0].item()], [0.0, 0.0], s=32, color=COLORS["dark"], zorder=7)

# Keep c as the lens-plane height marker inside the lens, matching the book.
ax.plot([0.0, 0.0], [0.0, c.item()], color=COLORS["gray"], lw=0.9, ls=(0, (3, 2)), zorder=4)
ax.text(0.14, c.item() - 0.02, r"$c$", fontsize=12, color=COLORS["dark"], ha="left", va="center")

# Theta_1 at the object end.
x1 = obj[0] + arc_r * torch.cos(left_arc)
y1 = arc_r * torch.sin(left_arc)
ax.plot([v.item() for v in x1], [v.item() for v in y1], color=COLORS["dark"], lw=1.0, zorder=4)
mid1 = theta1 / 2
txt1 = obj + (arc_r + torch.tensor(0.18, dtype=torch.float32)) * torch.stack([torch.cos(mid1), torch.sin(mid1)])
ax.text(txt1[0].item(), txt1[1].item(), r"$\theta_1$", fontsize=12, color=COLORS["dark"], ha="left", va="center")

# Theta_4 at the image end.
x4 = img[0] + arc_r * torch.cos(right_arc)
y4 = arc_r * torch.sin(right_arc)
ax.plot([v.item() for v in x4], [v.item() for v in y4], color=COLORS["dark"], lw=1.0, zorder=4)
mid4 = torch.pi - theta4 / 2
txt4 = img + (arc_r + torch.tensor(0.18, dtype=torch.float32)) * torch.stack([torch.cos(mid4), torch.sin(mid4)])
ax.text(txt4[0].item(), txt4[1].item(), r"$\theta_4$", fontsize=12, color=COLORS["dark"], ha="right", va="center")

# Distance brackets stop at the lens, not at a point behind it.
for x0, x1, label in [
    (obj[0], front_vertex[0], r"$a$"),
    (back_vertex[0], img[0], r"$b$"),
]:
    ax.plot([x0.item(), x1.item()], [br_y.item(), br_y.item()], color=COLORS["red"], lw=1.3, zorder=3)
    ax.plot([x0.item(), x0.item()], [(br_y - tick).item(), (br_y + tick).item()], color=COLORS["red"], lw=1.3, zorder=3)
    ax.plot([x1.item(), x1.item()], [(br_y - tick).item(), (br_y + tick).item()], color=COLORS["red"], lw=1.3, zorder=3)
    ax.text((0.5 * (x0 + x1)).item(), (br_y + 0.12).item(), label, fontsize=12, color=COLORS["red"], ha="center", va="bottom")

ax.text((0.5 * (img[0] - a)).item(), -1.60, "(a)", fontsize=11, style="italic", color=COLORS["dark"], ha="center", va="center")

ax.set_xlim((-a - 0.55).item(), (b + 0.55).item())
ax.set_ylim(-1.85, 1.35)
ax.axis("off")

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_04a.png", dpi=150, bbox_inches="tight")
plt.show()

The figure depicts an idealization: rays appear to bend at a single point (the center of the lens) rather than refracting twice (at each glass surface). This is the *thin-lens approximation*, which assumes the lens's physical thickness is small enough to ignore. Panel (b), drawn next, distorts the geometry to expose the actual two-surface bending that the thin-lens approximation glosses over.

**Figure 6.4(b) — The labeled thin-lens geometry.** Panel (b) follows a single ray through the same thin lens as panel (a), with the geometry deliberately distorted — two surfaces pulled apart, angles enlarged — so every label stays legible. The ray refracts at the front surface, crosses the glass, refracts again at the back, and meets the axis at the image point, turning through the four angles $\theta_1, \theta_2, \theta_3, \theta_4$. At each surface, the local tilt $\theta_S$ is set by the height $c$ at which the ray crosses (brackets $C_1 \approx C_2 \approx C$ because the lens is thin, $d \approx 0$). The object distance $a$, image distance $b$, and negligible thickness $d \approx 0$ are marked below the axis. The paraxial Snell's law at each surface gives the two relations the derivation sums:

$$n\,\theta_2 = \theta_1 + \theta_S, \qquad n\,\theta_3 = \theta_4 + \theta_S.$$

(Book §6.2, Figure 6.4(b) and Table 6.1.)

In [ ]:
# Figure 6.4 (b) — Thin-lens angle geometry (book §6.2).
#
# This cell draws the book-style lens diagram.
# The parameters below drive the ray path, surface normals, distance brackets,
# and angle bookkeeping; changing a value and rerunning redraws the figure
# from the updated geometry.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
n = torch.tensor(1.5, dtype=torch.float32)
R1 = torch.tensor(1.4, dtype=torch.float32)
R2 = torch.tensor(1.4, dtype=torch.float32)
c = torch.tensor(1.0, dtype=torch.float32)
theta1_deg = torch.tensor(15.0, dtype=torch.float32)
exaggeration = torch.tensor(5.0, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
lens_bulge = torch.tensor(0.27, dtype=torch.float32)     # how much the lens body bulges horizontally
lens_half_height = torch.tensor(2.42, dtype=torch.float32)  # vertical half-extent of the lens body
ray_height_factor = torch.tensor(1.34, dtype=torch.float32)  # visible ray height as a multiple of c

# --- Derived quantities ---
zero = torch.tensor(0.0, dtype=torch.float32)
pi = torch.tensor(torch.pi, dtype=torch.float32)

theta1_axis, theta2_axis, theta4_axis, thetaS1, thetaS2 = lensmaker_angle_bookkeeping(
    n, R1, R2, c, theta1_deg
)

gap = torch.tensor(0.12, dtype=torch.float32) * exaggeration
bulge = lens_bulge
half_h = lens_half_height
ray_h = ray_height_factor * c

front_vertex = torch.stack([-gap, zero])
back_vertex = torch.stack([gap, zero])

front_x = surface_x_at_y(ray_h, gap, bulge, half_h, side="left")
front_hit = torch.stack([front_x, ray_h])
back_hit = solve_surface_hit_from_ray(front_hit, theta2_axis, gap, bulge, half_h)

object_x = front_hit[0] - front_hit[1] / torch.tan(theta1_axis)
image_x = back_hit[0] - back_hit[1] / torch.tan(theta4_axis)

obj = torch.stack([object_x, zero])
img = torch.stack([image_x, zero])

left_x, right_x, lens_y = lens_curves(gap, bulge, half_h)

front_normal_air = pi - thetaS1
front_normal_glass = front_normal_air - pi
back_normal_air = thetaS2
back_normal_glass = back_normal_air + pi

front_ref_left = front_hit - torch.stack([torch.tensor(1.55, dtype=torch.float32), zero])
back_ref_right = back_hit + torch.stack([torch.tensor(1.55, dtype=torch.float32), zero])

C1_x = front_hit[0] - torch.tensor(0.78, dtype=torch.float32)
C2_x = back_hit[0] + torch.tensor(0.78, dtype=torch.float32)

a_y = torch.tensor(-2.37, dtype=torch.float32)
b_y = torch.tensor(-2.37, dtype=torch.float32)
d_y = torch.tensor(-1.72, dtype=torch.float32)
tick = torch.tensor(0.11, dtype=torch.float32)

dark, gray, blue, red, green, normal_color = (
    COLORS["dark"], COLORS["gray"], COLORS["blue"], COLORS["red"], COLORS["green"], COLORS["normal"]
)

# --- Drawing ---
fig = plt.figure(figsize=(12.0, 6.2))
ax = fig.add_axes([0.04, 0.10, 0.92, 0.82])

ax.axhline(0.0, color=dark, lw=1.05, zorder=0)
draw_biconvex_lens(ax, left_x, right_x, lens_y, face="#dbe7f5", lw=1.45, alpha=0.92)

# Draw the ray as one continuous path with bends at the two surfaces.
# The first two segments are plain lines; only the final segment carries
# the arrowhead to mark the propagation direction at the exit side.
ax.plot(
    [obj[0].item(), front_hit[0].item()],
    [obj[1].item(), front_hit[1].item()],
    color=blue,
    lw=1.55,
    zorder=5,
)
ax.plot(
    [front_hit[0].item(), back_hit[0].item()],
    [front_hit[1].item(), back_hit[1].item()],
    color=blue,
    lw=1.55,
    zorder=5,
)
ax.add_patch(
    FancyArrowPatch(
        (back_hit[0].item(), back_hit[1].item()),
        (img[0].item(), img[1].item()),
        arrowstyle="-|>",
        mutation_scale=9,
        linewidth=1.55,
        color=blue,
        shrinkA=0,
        shrinkB=0,
        zorder=5,
    )
)

# Horizontal reference directions at the two hit heights mark directions parallel to the axis.
ax.plot(
    [front_ref_left[0].item(), front_hit[0].item()],
    [front_hit[1].item(), front_hit[1].item()],
    color=dark,
    lw=0.85,
    zorder=2,
)
ax.plot(
    [back_hit[0].item(), back_ref_right[0].item()],
    [back_hit[1].item(), back_hit[1].item()],
    color=dark,
    lw=0.85,
    zorder=2,
)
ax.text(
    front_ref_left[0].item() + 0.05,
    front_hit[1].item() + 0.13,
    "parallel to optical axis",
    fontsize=8.8,
    color=gray,
)
ax.text(
    back_hit[0].item() + 0.15,
    back_hit[1].item() + 0.13,
    "parallel to optical axis",
    fontsize=8.8,
    color=gray,
)

# Surface normals pass through the hit points and define the local surface orientation.
for hit, normal_from, normal_to in [
    (front_hit, front_normal_air, front_normal_glass),
    (back_hit, back_normal_glass, back_normal_air),
]:
    normal_p0 = hit + torch.tensor(1.25, dtype=torch.float32) * torch.stack(
        [torch.cos(normal_from), torch.sin(normal_from)]
    )
    normal_p1 = hit + torch.tensor(1.25, dtype=torch.float32) * torch.stack(
        [torch.cos(normal_to), torch.sin(normal_to)]
    )
    ax.plot(
        [normal_p0[0].item(), normal_p1[0].item()],
        [normal_p0[1].item(), normal_p1[1].item()],
        color=normal_color,
        lw=1.0,
        zorder=4,
    )

# Small right-angle marks indicate that each normal is perpendicular to the local surface.
front_tangent = front_normal_air - 0.5 * pi
back_tangent = back_normal_air + 0.5 * pi
for hit, normal_air, tangent in [
    (front_hit, front_normal_air, front_tangent),
    (back_hit, back_normal_air, back_tangent),
]:
    mark_a = hit + torch.tensor(0.18, dtype=torch.float32) * torch.stack(
        [torch.cos(tangent), torch.sin(tangent)]
    )
    mark_b = mark_a + torch.tensor(0.18, dtype=torch.float32) * torch.stack(
        [torch.cos(normal_air), torch.sin(normal_air)]
    )
    mark_c = hit + torch.tensor(0.18, dtype=torch.float32) * torch.stack(
        [torch.cos(normal_air), torch.sin(normal_air)]
    )
    ax.plot(
        [mark_a[0].item(), mark_b[0].item(), mark_c[0].item()],
        [mark_a[1].item(), mark_b[1].item(), mark_c[1].item()],
        color=normal_color,
        lw=0.8,
        zorder=6,
    )

# The red height brackets show the two surface ray heights that are approximated by one common height.
for x_pos, y_top, text, ha, dx in [
    (C1_x, front_hit[1], r"$C_1 \approx C$", "right", -0.08),
    (C2_x, back_hit[1], r"$C_2 \approx C$", "left", 0.08),
]:
    ax.plot([x_pos.item(), x_pos.item()], [0.0, y_top.item()], color=red, lw=1.05, zorder=3)
    ax.plot([x_pos.item() - 0.08, x_pos.item() + 0.08], [0.0, 0.0], color=red, lw=1.05, zorder=3)
    ax.plot([x_pos.item() - 0.08, x_pos.item() + 0.08], [y_top.item(), y_top.item()], color=red, lw=1.05, zorder=3)
    ax.text(
        x_pos.item() + dx,
        (0.5 * y_top).item(),
        text,
        fontsize=11.0,
        color=red,
        ha=ha,
        va="center",
    )

# The lower brackets mark the object distance, lens thickness, and image distance.
for x0, x1, y0, text, color, y_text in [
    (obj[0], front_vertex[0], a_y, r"$a$", red, a_y - 0.20),
    (back_vertex[0], img[0], b_y, r"$b$", red, b_y - 0.20),
    (front_vertex[0], back_vertex[0], d_y, r"$d \approx 0$", red, d_y - 0.18),
]:
    ax.plot([x0.item(), x1.item()], [y0.item(), y0.item()], color=color, lw=1.0, zorder=3)
    ax.plot([x0.item(), x0.item()], [(y0 - tick).item(), (y0 + tick).item()], color=color, lw=1.0, zorder=3)
    ax.plot([x1.item(), x1.item()], [(y0 - tick).item(), (y0 + tick).item()], color=color, lw=1.0, zorder=3)
    ax.text(
        (0.5 * (x0 + x1)).item(),
        y_text.item(),
        text,
        fontsize=12.0,
        color=color,
        ha="center",
        va="center",
    )

# Draw the curved angle arrows that remain readable in the full-lens view.
main_angle_specs = [
    (obj, zero, theta1_axis, torch.tensor(0.55), r"$\theta_1$", torch.tensor(0.76), torch.tensor([0.04, 0.02], dtype=torch.float32)),
    (front_hit, pi, pi + theta1_axis, torch.tensor(0.48), r"$\theta_1$", torch.tensor(0.68), torch.tensor([-0.02, -0.02], dtype=torch.float32)),
    (front_hit, pi, front_normal_air, torch.tensor(0.72), r"$\theta_S$", torch.tensor(0.92), torch.tensor([-0.08, 0.08], dtype=torch.float32)),
    (back_hit, zero, theta4_axis, torch.tensor(0.48), r"$\theta_4$", torch.tensor(0.70), torch.tensor([0.04, -0.03], dtype=torch.float32)),
    (back_hit, zero, back_normal_air, torch.tensor(0.72), r"$\theta_S$", torch.tensor(0.93), torch.tensor([0.08, 0.08], dtype=torch.float32)),
    (img, pi + theta4_axis, pi, torch.tensor(0.55), r"$\theta_4$", torch.tensor(0.76), torch.tensor([-0.04, 0.03], dtype=torch.float32)),
]

for center, angle_a, angle_b, radius, text, text_radius, text_shift in main_angle_specs:
    delta = torch.atan2(torch.sin(angle_b - angle_a), torch.cos(angle_b - angle_a))
    a0 = angle_a
    a1 = angle_a + delta

    start = center + radius * torch.stack([torch.cos(a0), torch.sin(a0)])
    end = center + radius * torch.stack([torch.cos(a1), torch.sin(a1)])
    rad = 0.22 if delta.item() >= 0.0 else -0.22

    ax.add_patch(
        FancyArrowPatch(
            (start[0].item(), start[1].item()),
            (end[0].item(), end[1].item()),
            connectionstyle=f"arc3,rad={rad}",
            arrowstyle="-|>",
            mutation_scale=8,
            linewidth=0.9,
            color=green,
            shrinkA=0,
            shrinkB=0,
            zorder=7,
        )
    )

    mid_angle = 0.5 * (a0 + a1)
    text_point = center + text_radius * torch.stack([torch.cos(mid_angle), torch.sin(mid_angle)]) + text_shift
    ax.text(
        text_point[0].item(),
        text_point[1].item(),
        text,
        fontsize=10.2,
        color=green,
        ha="center",
        va="center",
    )

# Draw theta_2 as a leader annotation.
theta2_point = front_hit + torch.tensor([0.30, -0.015], dtype=torch.float32)
theta2_label = front_hit + torch.tensor([0.82, -0.36], dtype=torch.float32)

ax.annotate(
    r"$\theta_2$",
    xy=(theta2_point[0].item(), theta2_point[1].item()),
    xytext=(theta2_label[0].item(), theta2_label[1].item()),
    fontsize=10.4,
    color=green,
    ha="center",
    va="center",
    arrowprops=dict(
        arrowstyle="-|>",
        color=green,
        lw=1.0,
        shrinkA=2,
        shrinkB=2,
        connectionstyle="arc3,rad=-0.15",
    ),
    zorder=8,
)

# Draw theta_3 as a leader annotation.
theta3_point = back_hit + torch.tensor([-0.30, 0.015], dtype=torch.float32)
theta3_label = back_hit + torch.tensor([-0.82, 0.20], dtype=torch.float32)

ax.annotate(
    r"$\theta_3$",
    xy=(theta3_point[0].item(), theta3_point[1].item()),
    xytext=(theta3_label[0].item(), theta3_label[1].item()),
    fontsize=10.4,
    color=green,
    ha="center",
    va="center",
    arrowprops=dict(
        arrowstyle="-|>",
        color=green,
        lw=1.0,
        shrinkA=2,
        shrinkB=2,
        connectionstyle="arc3,rad=0.15",
    ),
    zorder=8,
)

ax.scatter([obj[0].item(), img[0].item()], [0.0, 0.0], s=28, color="black", zorder=8)

ax.text(-2.5, 0.15, "optical axis", fontsize=10.0, color=gray, ha="left", va="bottom")
ax.text(-3.75, 2.20, r"air, $n=1$", fontsize=12, color=gray)
ax.text(0.0, -1.15, r"$n$", fontsize=14, color=dark, ha="center")

ax.text(
    2.5,
    -2.70,
    "distances and angles distorted for clarity of labeling",
    fontsize=9.2,
    color="#4d74a6",
    style="italic",
)
ax.text(0.0, -2.70, r"$(b)$", fontsize=13, color=dark, ha="center", style="italic")

ax.set_xlim(min(-5.6, obj[0].item() - 0.35), max(5.6, img[0].item() + 0.35))
ax.set_ylim(-2.88, 2.68)
ax.set_aspect("equal", adjustable="box")
ax.axis("off")

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_04b.png", dpi=150, bbox_inches="tight")
plt.show()

**From the diagram to the lensmaker's formula.** The book's Table 6.1 sums the four small-angle Snell relations along the ray's path. Substituting the axis angles $\theta_1 \approx c/a$ and $\theta_4 \approx c/b$, and the surface tilts $\theta_{S_1} \approx c/R_1$ and $\theta_{S_2} \approx c/R_2$, gives

$$\frac{1}{a} + \frac{1}{b} = (n-1)\left(\frac{1}{R_1} + \frac{1}{R_2}\right).$$

The height $c$ cancels on both sides — every ray from the same object point lands at the same image distance $b$, regardless of where it crosses the lens. Defining

$$\frac{1}{f} = (n-1)\left(\frac{1}{R_1} + \frac{1}{R_2}\right)$$

gives the lensmaker's formula in the target form:

$$\frac{1}{a} + \frac{1}{b} = \frac{1}{f}.$$

(Book §6.2, Figure 6.4(b) and Table 6.1.)

### From flat interface to curved surface

The angles in the lensmaker derivation are measured from each lens surface's *normal*, but the lens's surfaces aren't flat — they're spherical. Two things change with curvature. The normal at the point where a ray strikes the surface is no longer parallel to the optical axis, and the tilt of that normal depends on how far above the axis the ray hits. The next figure pins down that dependence: how the surface's tilt angle $\theta_S$ relates to the radius of curvature $R$ and the hit height $c$.

**Figure 6.5 — Relation between $R$ and $\theta_S$.** A spherical surface of radius $R$ is drawn as a full circle centered on the optical axis. A ray meets the surface at height $c$ above the axis. The slanted radius from the center to that hit point makes an angle $\theta_S$ with the horizontal axis-radius — and the same angle reappears at the surface, between the vertical reference direction and the surface normal. In the small triangle formed by the slanted radius, the horizontal axis-radius, and the vertical leg of height $c$, basic trigonometry gives $\sin\theta_S = c/R$. In the paraxial regime, this simplifies to

$$\theta_S \approx \frac{c}{R},$$

which is exactly what the `surface_angle` helper above returns, and what the lensmaker derivation in Figure 6.4(b) used at each surface. (Book §6.2, Figure 6.5.)

In [ ]:
# Figure 6.5 — Relation between R and theta_S (book §6.2).
#
# A spherical surface hit at height c defines the normal tilt theta_S through R.
# Changing c or R recomputes the hit point, radius triangle, and echoed angle.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
R = torch.tensor(1.0, dtype=torch.float32)
c = torch.tensor(0.30, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
theta_scale = torch.tensor(1.15, dtype=torch.float32)
pad = torch.tensor(0.10, dtype=torch.float32)
dark = COLORS["dark"]

# --- Derived quantities ---
z = torch.tensor(0.0, dtype=torch.float32)
pi = torch.tensor(torch.pi, dtype=torch.float32)
theta_s = theta_scale * surface_angle(c, R)

center = torch.stack([z, z])
axis_left = torch.stack([-R, z])
hit = center + R * torch.stack([-torch.cos(theta_s), torch.sin(theta_s)])
vertical_top = hit + torch.stack([z, torch.tensor(0.70, dtype=torch.float32)])
normal_top = hit + torch.tensor(0.72, dtype=torch.float32) * torch.stack(
    [-torch.sin(theta_s), torch.cos(theta_s)]
)

t = torch.linspace(0.0, 2.0 * pi, 400)
circle = torch.stack([R * torch.cos(t), R * torch.sin(t)], dim=1)

# --- Drawing ---
fig, ax = plt.subplots(figsize=(6.2, 6.2))

ax.plot([p[0].item() for p in circle], [p[1].item() for p in circle], color=dark, lw=1.2)

for p0, p1, lw in [
    (axis_left, center, 2.0),
    (hit, center, 2.0),
    (torch.stack([hit[0], z]), hit, 2.0),
    (hit, vertical_top, 2.0),
    (hit, normal_top, 2.0),
]:
    ax.plot([p0[0].item(), p1[0].item()], [p0[1].item(), p1[1].item()], color=dark, lw=lw)

for args in [
    (center, pi, pi - theta_s, torch.tensor(0.38), r"$\theta_S$", torch.tensor(0.56), torch.stack([z, -torch.tensor(0.02)])),
    (hit, 0.5 * pi, 0.5 * pi - theta_s, torch.tensor(0.34), r"$\theta_S$", torch.tensor(0.50), torch.stack([-torch.tensor(0.08), torch.tensor(0.02)])),
]:
    draw_angle(ax, *args, fs=20, color=dark, lw=1.0)

for p, text, fs, ha, va in [
    (0.55 * (hit + center) + torch.tensor([0.0, 0.06]), r"$R$", 22, "center", "center"),
    (torch.stack([hit[0] + torch.tensor(0.08), 0.5 * hit[1]]), r"$c$", 22, "left", "center"),
]:
    ax.text(p[0].item(), p[1].item(), text, fontsize=fs, color=dark, ha=ha, va=va)

ax.set_xlim((-R - pad).item(), (R + pad).item())
ax.set_ylim((-R - pad).item(), (R + pad).item())
ax.set_aspect("equal", adjustable="box")
ax.axis("off")

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_05.png", dpi=150, bbox_inches="tight")
plt.show()

With $\theta_S \approx c/R$ established, the surface-tilt relation above can drop $c$ on both sides — completing the cancellation that lets one image distance $b$ serve every ray from the same object point.

### Off-axis points

The derivation so far placed the object on the optical axis. Off-axis points need only a small extension: rotating the whole construction through a small angle $\theta_R$ adds $\theta_R$ to $\theta_S$ at each surface in Table 6.1, leaving the algebra unchanged. The lensmaker's formula still holds, and the image lands at the conjugate distance $b$ on the far side, at height $P_2 = -b\,P_1 / a$ — inverted, below the axis.

In the paraxial, thin-lens limit, this promotes the focusing property from points on an axis to whole planes: every point on the object plane at distance $a$ focuses to a corresponding point on the image plane at distance $b$, both perpendicular to the optical axis.

**Figure 6.6 — Off-axis points and the equivalent lens rotation.** The off-axis object point $P_1$ sits at height $P_1$ above the optical axis, at distance $a$ from the lens; $P_0$ marks the on-axis reference. Two rays trace the image — a parallel ray refracts through the focal point at distance $f$, and a ray through the lens center proceeds undeviated. They meet at the image point on the image plane at distance $b$. The angle $\theta_R$ marks the small rotation that makes the off-axis case equivalent to the on-axis derivation. (Book §6.2, Figure 6.6.)

In [ ]:
# Figure 6.6 — Off-axis points and the equivalent lens rotation (book §6.2).
#
# This cell draws one off-axis object point, the chief ray through the lens
# center, and the parallel ray that refracts to the image. Changing n, R1, R2,
# a, or P1 updates f, b, P2, and theta_R.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
n = torch.tensor(1.5, dtype=torch.float32)
R1 = torch.tensor(1.4, dtype=torch.float32)
R2 = torch.tensor(1.4, dtype=torch.float32)
a = torch.tensor(4.0, dtype=torch.float32)
P1 = torch.tensor(1.05, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
lens_gap = torch.tensor(0.0, dtype=torch.float32)
lens_bulge = torch.tensor(0.045, dtype=torch.float32)
lens_half_h = torch.tensor(1.75, dtype=torch.float32)
tick = torch.tensor(0.06, dtype=torch.float32)
dark, blue, red, gray, green = (
    COLORS["dark"], COLORS["blue"], COLORS["red"], COLORS["gray"], COLORS["green"]
)

# --- Derived quantities ---
z = torch.tensor(0.0, dtype=torch.float32)
f = focal_length(n, R1, R2)
b = image_distance(a, f)
theta_R = torch.atan(P1 / a)
P2 = -b * torch.tan(theta_R)

obj = torch.stack([-a, P1])
obj0 = torch.stack([-a, z])
lens0 = torch.stack([z, z])
lens1 = torch.stack([z, P1])
img = torch.stack([b, P2])

left_x, right_x, lens_y = lens_curves(lens_gap, lens_bulge, lens_half_h)
left_vertex = surface_x_at_y(z, lens_gap, lens_bulge, lens_half_h, side="left")
right_vertex = surface_x_at_y(z, lens_gap, lens_bulge, lens_half_h, side="right")

a_y = torch.tensor(-0.14, dtype=torch.float32)
f_y = torch.tensor(-0.05, dtype=torch.float32)
b_y = torch.tensor(-0.92, dtype=torch.float32)
ymin = torch.min(P2 - torch.tensor(0.28, dtype=torch.float32), b_y - torch.tensor(0.12, dtype=torch.float32))
ymax = P1 + torch.tensor(0.22, dtype=torch.float32)

# --- Drawing ---
fig, ax = plt.subplots(figsize=(12.8, 4.0))

ax.plot([(-a).item(), b.item()], [0.0, 0.0], color=gray, lw=0.8, zorder=0)
draw_biconvex_lens(ax, left_x, right_x, lens_y, face="white", lw=1.5, alpha=1.0)

for x, y0, y1 in [(-a, ymin, ymax), (b, ymin, torch.tensor(0.84, dtype=torch.float32))]:
    ax.plot([x.item(), x.item()], [y0.item(), y1.item()], color=dark, lw=1.1)

ax.plot([obj[0].item(), lens1[0].item()], [obj[1].item(), lens1[1].item()], color=blue, lw=1.4)
ax.plot([lens1[0].item(), img[0].item()], [lens1[1].item(), img[1].item()], color=blue, lw=1.4)
ax.plot([obj[0].item(), img[0].item()], [obj[1].item(), img[1].item()], color=dark, lw=0.8, ls=":")

# Anchor the distance bars at the lens surfaces, not the lens center.
for x0, x1, y, label, dy in [
    (-a, left_vertex, a_y, "a", -0.12),
    (right_vertex, f, f_y, "f", -0.18),
    (right_vertex, b, b_y, "b", -0.12),
]:
    ax.plot([x0.item(), x1.item()], [y.item(), y.item()], color=red, lw=1.3)
    ax.plot([x0.item(), x0.item()], [(y - tick).item(), (y + tick).item()], color=red, lw=1.3)
    ax.plot([x1.item(), x1.item()], [(y - tick).item(), (y + tick).item()], color=red, lw=1.3)
    ax.text((0.5 * (x0 + x1)).item(), (y + dy).item(), label, fontsize=16, color=dark, ha="center", va="center")

draw_angle(
    ax,
    lens0,
    torch.pi - theta_R,
    torch.pi,
    torch.tensor(0.82, dtype=torch.float32),
    r"$\theta_R$",
    text_radius=torch.tensor(0.98, dtype=torch.float32),
    text_shift=torch.tensor([-0.05, 0.06], dtype=torch.float32),
    fs=16,
    color=green,
    lw=1.1,
)

for p, text, fs, ha, va in [
    (torch.stack([obj0[0] - torch.tensor(0.10), obj[1] + torch.tensor(0.02)]), r"$P_1$", 24, "right", "center"),
    (torch.stack([obj0[0] - torch.tensor(0.10), obj0[1] - torch.tensor(0.02)]), r"$P_0$", 24, "right", "center"),
    (torch.stack([right_vertex + torch.tensor(0.12), ymin - torch.tensor(0.02)]), "thin lens", 14, "left", "top"),
]:
    ax.text(p[0].item(), p[1].item(), text, fontsize=fs, color=dark, ha=ha, va=va)

ax.set_xlim((-a - torch.tensor(0.22)).item(), (b + torch.tensor(0.18)).item())
ax.set_ylim((ymin - torch.tensor(0.18)).item(), (ymax + torch.tensor(0.10)).item())
ax.axis("off")

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_06.png", dpi=150, bbox_inches="tight")
plt.show()

With Figure 6.6, §6.2 is complete: the lensmaker's formula

$$\frac{1}{a} + \frac{1}{b} = \frac{1}{f}, \qquad \frac{1}{f} = (n-1)\left(\frac{1}{R_1} + \frac{1}{R_2}\right),$$

describes how a thin lens maps every point on an object plane at distance $a$ to a corresponding point on the image plane at distance $b$. The book's Figure 6.7, a photograph of a laser pointer swept across a lens with every ray converging to the same spot on the wall, is the same focusing property demonstrated physically; that photo isn't reproduced here.

The next section takes the lensmaker's formula and puts it to work — straight-through rays at the lens center, conjugate-point ray tracing, depth of field, concave lenses, and a Galilean telescope built from two of them.

## 6.3 — Imaging with Lenses

With the lensmaker's formula in hand, the rest of the chapter puts it to work — predicting where images form, how sharp they are, what changes with a concave lens, and how two lenses combine into a telescope. The starting observation is a small one, but it's what lets the whole apparatus mimic a pinhole camera.

At the very center of a thin lens, the front and back surfaces are parallel. A ray crossing that region refracts at the first surface, then refracts back through the same angle at the second surface — emerging parallel to the way it entered, displaced laterally by a small amount that depends on the lens's thickness. In the *thin*-lens limit, the two surfaces collapse to a single plane: the displacement vanishes and the ray passes straight through. Every direction through the center is undeviated, which means the lens behaves, for those center rays, exactly like a pinhole — and a thin lens therefore images the world in *perspective projection*, just as a pinhole does.

**Figure 6.8 — Rays through the center of a thin lens.** **(a)** A physical lens of non-zero thickness: the ray refracts at both surfaces and exits parallel to the incoming direction, with a small lateral displacement. **(b)** The same ray under the thin-lens approximation: the two surfaces collapse to a single plane, the displacement vanishes, and the ray passes straight through. **(c)** Because *every* direction through the center is undeviated, a fan of rays through the lens center behaves identically to a fan of rays through a pinhole. (Book §6.3, Figure 6.8.)

In [ ]:
# Figure 6.8 (a–c) — Center ray through a physical lens and its thin-lens limit (book §6.3).
#
# Panels (a) and (b) compare the center ray in a thick lens and its thin-lens surrogate.
# Panel (c) shows a family of rays that pass through the thin-lens center undeviated.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
n_air = torch.tensor(1.0, dtype=torch.float32)
n_glass = torch.tensor(1.5, dtype=torch.float32)
theta1_deg = torch.tensor(24.0, dtype=torch.float32)
physical_thickness = torch.tensor(0.72, dtype=torch.float32)
fan_half_angle_deg = torch.tensor(22.0, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
ray_len_left = torch.tensor(1.55, dtype=torch.float32)
ray_len_right = torch.tensor(1.65, dtype=torch.float32)
fan_len = torch.tensor(1.75, dtype=torch.float32)

physical_lens_half_h = torch.tensor(1.02, dtype=torch.float32)
thin_lens_half_h = torch.tensor(0.92, dtype=torch.float32)
thin_lens_gap = torch.tensor(0.018, dtype=torch.float32)
thin_lens_bulge = torch.tensor(0.030, dtype=torch.float32)

dark, blue, cyan, panel_gray = (
    COLORS["dark"], COLORS["blue"], COLORS["cyan"], COLORS["panel"]
)

# --- Derived quantities ---
zero = torch.tensor(0.0, dtype=torch.float32)
pi = torch.tensor(torch.pi, dtype=torch.float32)

theta1 = torch.deg2rad(theta1_deg)
phi1 = 0.5 * pi - theta1
phi2 = snells_law(n_air, n_glass, phi1)
theta_glass = 0.5 * pi - phi2

dir_air = torch.stack([torch.sin(theta1), -torch.cos(theta1)])
dir_glass = torch.stack([torch.sin(theta_glass), -torch.cos(theta_glass)])

x_left = -0.5 * physical_thickness
x_right = 0.5 * physical_thickness
hit_left = torch.stack([x_left, torch.tensor(0.18, dtype=torch.float32)])
hit_right = hit_left + ((x_right - x_left) / dir_glass[0]) * dir_glass

src_a = hit_left - ray_len_left * dir_air
dst_a = hit_right + ray_len_right * dir_air

center_b = torch.stack([zero, zero])
src_b = center_b - ray_len_left * dir_air
dst_b = center_b + ray_len_right * dir_air

fan_angles = torch.deg2rad(torch.linspace(-fan_half_angle_deg, fan_half_angle_deg, 5, dtype=torch.float32))
fan_left = [torch.stack([zero, zero]) - fan_len * torch.stack([torch.cos(a), torch.sin(a)]) for a in fan_angles]
fan_right = [torch.stack([zero, zero]) + fan_len * torch.stack([torch.cos(a), torch.sin(a)]) for a in fan_angles]

left_x, right_x, lens_y = lens_curves(thin_lens_gap, thin_lens_bulge, thin_lens_half_h)

ray_rot = torch.rad2deg(torch.atan2(dir_air[1], dir_air[0])).item()
theta_box = dict(facecolor="white", edgecolor="none", pad=0.12, alpha=0.9)

# --- Drawing ---
fig, axs = plt.subplots(1, 3, figsize=(15.2, 5.0))

# (a) Center of physical lens.
ax = axs[0]
for x in [x_left, x_right]:
    ax.plot([x.item(), x.item()], [-0.95, 1.15], color=cyan, lw=1.6)
for p0, p1 in [(src_a, hit_left), (hit_left, hit_right), (hit_right, dst_a)]:
    ax.plot([p0[0].item(), p1[0].item()], [p0[1].item(), p1[1].item()], color=blue, lw=2.4)
ax.text(0.0, 1.35, "Center of\nphysical lens", ha="center", va="center", fontsize=16)
ax.text((src_a[0] + 0.18).item(), (src_a[1] + 0.08).item(), "ray direction", rotation=ray_rot, fontsize=13, color=dark)
ax.text((hit_left[0] - 0.18).item(), (0.5 * (hit_left[1] + src_a[1]) + 0.06).item(), r"$\theta_1$", fontsize=18, color=dark, bbox=theta_box)
ax.text((hit_right[0] + 0.02).item(), (0.5 * (hit_right[1] + dst_a[1]) - 0.06).item(), r"$\theta_1$", fontsize=18, color=dark, bbox=theta_box)
ax.text(0.0, -1.48, "(a)", ha="center", va="center", fontsize=20, color=panel_gray, style="italic")
ax.set_xlim(-1.65, 1.85)
ax.set_ylim(-1.55, 1.55)
ax.axis("off")

# (b) Center of thin lens.
ax = axs[1]
ax.plot([0.0, 0.0], [-0.95, 1.15], color=cyan, lw=1.8)
ax.plot([src_b[0].item(), dst_b[0].item()], [src_b[1].item(), dst_b[1].item()], color=blue, lw=2.4)
ax.text(0.0, 1.35, "Center of thin\nlens", ha="center", va="center", fontsize=16)
ax.text((src_b[0] + 0.18).item(), (src_b[1] + 0.08).item(), "ray direction", rotation=ray_rot, fontsize=13, color=dark)
ax.text(-0.42, 0.28, r"$\theta_1$", fontsize=18, color=dark, bbox=theta_box)
ax.text(0.12, -0.42, r"$\theta_1$", fontsize=18, color=dark, bbox=theta_box)
ax.text(0.0, -1.48, "(b)", ha="center", va="center", fontsize=20, color=panel_gray, style="italic")
ax.set_xlim(-1.65, 1.85)
ax.set_ylim(-1.55, 1.55)
ax.axis("off")

# (c) All rays through the thin-lens center.
ax = axs[2]
for p0, p1 in zip(fan_left, fan_right):
    ax.plot([p0[0].item(), p1[0].item()], [p0[1].item(), p1[1].item()], color=blue, lw=2.0)
draw_biconvex_lens(ax, left_x, right_x, lens_y, face="white", lw=1.4, alpha=1.0)
ax.text(0.0, 1.35, "thin lens", ha="center", va="center", fontsize=16)
ax.text(0.0, -1.48, "(c)", ha="center", va="center", fontsize=20, color=panel_gray, style="italic")
ax.set_xlim(-1.95, 1.95)
ax.set_ylim(-1.55, 1.55)
ax.axis("off")

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_08abc.png", dpi=150, bbox_inches="tight")
plt.show()

With this observation in place, four properties now characterize how rays travel through a thin lens:

1. **Focusing** (§6.2): every ray from a point at distance $a$ reconverges at the conjugate distance $b$, with $\frac{1}{a} + \frac{1}{b} = \frac{1}{f}$.
2. **Parallel rays**: the limit $a \to \infty$ — parallel rays converge at the focal point, distance $f$ behind the lens.
3. **Center rays**: any ray through the lens center proceeds in a straight line, as panel (c) above shows.
4. **Magnification**: combining the first and third, a plane at distance $a$ images with scale $b/a$ — exactly as a pinhole at the lens center would project it.

Two points on opposite sides of the lens at distances $a$ and $b$ that satisfy the lensmaker's formula are called **conjugate points**: light from one focuses to the other, and vice versa. The next figure traces conjugate-point pairs for several object distances, showing how the image distance moves as the object slides toward or away from the lens.

The four ray-tracing rules above pin down the *image distance* for any object distance through $\frac{1}{a} + \frac{1}{b} = \frac{1}{f}$. The two endpoints of the formula are limits — object at infinity, image at $f$; object at $f$, image at infinity — and everything in between trades smoothly between them. The next figure traces five cases on the same lens, holding the focal length fixed and moving the object closer step by step.

**Figure 6.9 — Conjugate points for a convex thin lens.** All five panels show the same lens with focal points marked at $\pm f$ from the lens (cyan dots). **(a)** Parallel rays from infinity converge at the right-side focal point — the limiting case $a \to \infty$, $b = f$. **(b)** Object at $a = 3f$ images at $b = 1.5f$. **(c)** Object at $a = 2f$ images symmetrically at $b = 2f$ — the unit-magnification case. **(d)** Object at $a = 1.5f$ images at $b = 3f$ — as the object moves inward, the image moves outward. **(e)** Object at the left focal point ($a = f$) produces parallel rays on exit — the other limiting case, $b \to \infty$. (Book §6.3, Figure 6.9.)

In [ ]:
# Figure 6.9 (a-e) — Conjugate points for a convex thin lens (book §6.3).
#
# This cell draws five standard conjugate-point cases for one thin lens.
# Changing n, R1, R2, or c updates f and the ray geometry across all panels.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
n = torch.tensor(1.5, dtype=torch.float32)
R1 = torch.tensor(1.4, dtype=torch.float32)
R2 = torch.tensor(1.4, dtype=torch.float32)
c = torch.tensor(0.55, dtype=torch.float32)

a_far_over_f = torch.tensor(3.0, dtype=torch.float32)
a_mid_over_f = torch.tensor(1.5, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
lens_gap = torch.tensor(0.0, dtype=torch.float32)
lens_bulge = torch.tensor(0.055, dtype=torch.float32)
lens_half_h_min = torch.tensor(0.85, dtype=torch.float32)
panel_pad_x = torch.tensor(0.45, dtype=torch.float32)
panel_pad_y = torch.tensor(0.30, dtype=torch.float32)
dot_size = 18
ray_lw = 1.1
lens_lw = 1.1

# --- Derived quantities ---
z = torch.tensor(0.0, dtype=torch.float32)
f = focal_length(n, R1, R2)

a_far = a_far_over_f * f
a_two = 2.0 * f
a_mid = a_mid_over_f * f

b_far = image_distance(a_far, f)
b_two = image_distance(a_two, f)
b_mid = image_distance(a_mid, f)

lens_half_h = torch.maximum(lens_half_h_min, 1.55 * c)
left_x, right_x, lens_y = lens_curves(lens_gap, lens_bulge, lens_half_h)

F_left = torch.stack([-f, z])
F_right = torch.stack([f, z])

cases = [
    {"tag": "(a)", "kind": "parallel"},
    {"tag": "(b)", "kind": "finite", "a": a_far, "b": b_far},
    {"tag": "(c)", "kind": "finite", "a": a_two, "b": b_two},
    {"tag": "(d)", "kind": "finite", "a": a_mid, "b": b_mid},
    {"tag": "(e)", "kind": "focus"},
]

# --- Drawing ---
fig, axs = plt.subplots(1, 5, figsize=(18, 4.4))
fig.subplots_adjust(wspace=0.22)

for ax, case in zip(axs, cases):
    ax.plot([-4.0, 4.0], [0.0, 0.0], color=COLORS["dark"], lw=0.9, zorder=0)
    draw_biconvex_lens(ax, left_x, right_x, lens_y, face="white", lw=lens_lw, alpha=1.0)

    ax.scatter(
        [F_left[0].item(), F_right[0].item()],
        [0.0, 0.0],
        s=dot_size,
        color=COLORS["cyan"],
        zorder=4,
    )

    if case["kind"] == "parallel":
        x0 = -2.35 * f
        img = F_right
        for y in [c, z, -c]:
            ax.plot([x0.item(), 0.0], [y.item(), y.item()], color=COLORS["blue"], lw=ray_lw, zorder=2)
            ax.plot([0.0, img[0].item()], [y.item(), img[1].item()], color=COLORS["blue"], lw=ray_lw, zorder=2)
        xlim = ((x0 - panel_pad_x).item(), (1.8 * f + panel_pad_x).item())

    elif case["kind"] == "focus":
        obj = F_left
        x1 = 2.35 * f
        for y in [c, z, -c]:
            ax.plot([obj[0].item(), 0.0], [obj[1].item(), y.item()], color=COLORS["blue"], lw=ray_lw, zorder=2)
            ax.plot([0.0, x1.item()], [y.item(), y.item()], color=COLORS["blue"], lw=ray_lw, zorder=2)
        xlim = ((-1.55 * f - panel_pad_x).item(), (x1 + panel_pad_x).item())

    else:
        obj = torch.stack([-case["a"], z])
        img = torch.stack([case["b"], z])
        for y in [c, z, -c]:
            ax.plot([obj[0].item(), 0.0], [obj[1].item(), y.item()], color=COLORS["blue"], lw=ray_lw, zorder=2)
            ax.plot([0.0, img[0].item()], [y.item(), img[1].item()], color=COLORS["blue"], lw=ray_lw, zorder=2)
        xlim = ((obj[0] - panel_pad_x).item(), (img[0] + panel_pad_x).item())

    ax.text(
        0.5, -0.14, case["tag"],
        transform=ax.transAxes,
        fontsize=20,
        color=COLORS["panel"],
        ha="center",
        va="top",
        style="italic",
    )

    ax.set_xlim(*xlim)
    ax.set_ylim((-(lens_half_h + panel_pad_y)).item(), (lens_half_h + panel_pad_y).item())
    ax.axis("off")

plt.savefig(IMAGES_DIR / "fig06_09.png", dpi=150, bbox_inches="tight")
plt.show()

Set the lens up inside a camera. The sensor sits at a fixed distance $b$ behind the lens; the object lives somewhere out in the world at distance $a$. If $a$ and $b$ happen to satisfy the lensmaker's formula, the object is in *focus* — every ray from a surface point converges to a single point on the sensor, and the image is sharp.

But $a$ varies with what the camera is pointed at, and $b$ is fixed by the camera's geometry. When $a$ doesn't quite match the conjugate of $b$, rays from a single object point don't converge at the sensor — they hit it as a small *circle of confusion*. How small can that circle get before the image looks blurry, and how much of the scene stays acceptably sharp at once? That's the **depth of field**, and it's what §6.3.1 derives next.

### 6.3.1 — Depth of Field

When the lens lives inside a camera, the sensor sits at a fixed distance behind it, so only one object distance is in sharp focus — the one whose conjugate is the sensor itself. That object distance defines an *object plane* (the *focal plane* in the book's terminology) where everything images sharply. Objects in front of or behind that plane image past or before the sensor, hitting it as a small disk — a *circle of confusion*. A photograph still looks sharp as long as that circle is small enough that the eye can't tell, and the range of object distances over which it stays acceptable is the *depth of field*.

**Figure 6.10 — Circle of confusion and depth of field.** Three object positions, one fixed sensor plane. The top row shows an object on the in-focus plane — every ray converges to a single point on the sensor for a sharp image. The middle row shows an object closer than the in-focus plane — its image forms past the sensor, hitting the sensor as a circle of confusion. The bottom row shows an object further than the in-focus plane — its image forms before the sensor, again hitting as a circle of confusion. The bracket on the left marks the depth of field. (Book §6.3.1, Figure 6.10.)

In [ ]:
# Figure 6.10 — Depth of field for a thin lens (book §6.3.1).
#
# This cell draws three object depths against one fixed sensor plane. The top
# row is in focus on the sensor; the lower rows miss that plane and produce a
# blur segment there.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
n = torch.tensor(1.5, dtype=torch.float32)
R1 = torch.tensor(1.4, dtype=torch.float32)
R2 = torch.tensor(1.4, dtype=torch.float32)
focus_a = torch.tensor(4.0, dtype=torch.float32)
near_a = torch.tensor(3.35, dtype=torch.float32)
far_a = torch.tensor(5.25, dtype=torch.float32)
aperture = torch.tensor(0.82, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
row_y = torch.tensor([2.55, 0.05, -2.45], dtype=torch.float32)
lens_gap = torch.tensor(0.0, dtype=torch.float32)
lens_bulge = torch.tensor(0.045, dtype=torch.float32)
lens_half_h = torch.tensor(0.84, dtype=torch.float32)
coc_dx = torch.tensor(0.18, dtype=torch.float32)
tick = torch.tensor(0.11, dtype=torch.float32)
dark, blue, cyan, gray = COLORS["dark"], COLORS["blue"], COLORS["cyan"], COLORS["gray"]

# --- Derived quantities ---
z = torch.tensor(0.0, dtype=torch.float32)
f = focal_length(n, R1, R2)
sensor_x = image_distance(focus_a, f)
depths = torch.stack([focus_a, near_a, far_a])
obj_x = -depths
img_x = image_distance(depths, f)
left_x, right_x, lens_y = lens_curves(lens_gap, lens_bulge, lens_half_h)


def sensor_intercept(hit, img, x_sensor):
    t = (x_sensor - hit[0]) / (img[0] - hit[0])
    return hit[1] + t * (img[1] - hit[1])


rows = []
for y0, x_obj, x_img in zip(row_y, obj_x, img_x):
    obj = torch.stack([x_obj, y0])
    top = torch.stack([z, y0 + aperture])
    bot = torch.stack([z, y0 - aperture])
    img = torch.stack([x_img, y0])
    y_top_s = sensor_intercept(top, img, sensor_x)
    y_bot_s = sensor_intercept(bot, img, sensor_x)
    rows.append({
        "obj": obj,
        "top": top,
        "bot": bot,
        "img": img,
        "y_lo": torch.minimum(y_top_s, y_bot_s),
        "y_hi": torch.maximum(y_top_s, y_bot_s),
    })

x_obj_plane = obj_x[0]
x_sensor_plane = sensor_x
x_min = obj_x.min() - torch.tensor(0.38, dtype=torch.float32)
x_max = sensor_x + torch.tensor(0.95, dtype=torch.float32)
y_min = row_y[-1] - torch.tensor(1.55, dtype=torch.float32)
y_max = row_y[0] + torch.tensor(1.35, dtype=torch.float32)

# --- Drawing ---
fig, ax = plt.subplots(figsize=(7.2, 8.2))

for x, label, ha in [(x_obj_plane, "object plane", "right"), (x_sensor_plane, "sensor plane", "left")]:
    ax.plot([x.item(), x.item()], [y_min.item(), y_max.item()], color=cyan, lw=0.8, alpha=0.45, zorder=0)
    ax.text(
        x.item(),
        (y_max - torch.tensor(0.18, dtype=torch.float32)).item(),
        label,
        fontsize=10, color=dark, ha=ha, va="bottom",
    )

for row, y0 in zip(rows, row_y):
    draw_biconvex_lens(ax, left_x, right_x, lens_y + y0, face="white", lw=1.2, alpha=1.0)
    for hit in [row["top"], row["bot"]]:
        y_sensor = sensor_intercept(hit, row["img"], x_sensor_plane).item()
        ax.plot(
            [row["obj"][0].item(), hit[0].item(), row["img"][0].item(), x_sensor_plane.item()],
            [row["obj"][1].item(), hit[1].item(), row["img"][1].item(), y_sensor],
            color=blue, lw=1.0, zorder=2,
        )

# Circle-of-confusion brackets for the two out-of-focus rows.
for row in rows[1:]:
    ax.plot(
        [x_sensor_plane.item(), x_sensor_plane.item()],
        [row["y_lo"].item(), row["y_hi"].item()],
        color=blue, lw=1.0, zorder=3,
    )
    bx = x_sensor_plane + coc_dx
    for y in [row["y_lo"], row["y_hi"]]:
        ax.plot([x_sensor_plane.item(), bx.item()], [y.item(), y.item()], color=gray, lw=0.8, ls=":")
    ax.plot([bx.item(), bx.item()], [row["y_lo"].item(), row["y_hi"].item()], color=gray, lw=0.8, ls=":")
    ax.text(
        (bx + torch.tensor(0.12, dtype=torch.float32)).item(),
        (0.5 * (row["y_lo"] + row["y_hi"])).item(),
        "circle of\nconfusion",
        fontsize=10, color=dark, ha="left", va="center",
    )

# Depth-of-field bracket below the rows.
dof_y = row_y[1] - torch.tensor(1.10, dtype=torch.float32)
near_x = rows[1]["obj"][0]
far_x = rows[2]["obj"][0]
for x0, y0 in [(far_x, rows[2]["obj"][1]), (near_x, rows[1]["obj"][1])]:
    ax.plot([x0.item(), x0.item()], [dof_y.item(), y0.item()], color=gray, lw=0.8, ls=":")
ax.plot([far_x.item(), near_x.item()], [dof_y.item(), dof_y.item()], color=dark, lw=0.9)
for x_end in [far_x, near_x]:
    ax.plot([x_end.item(), x_end.item()], [(dof_y - tick).item(), (dof_y + tick).item()], color=dark, lw=0.9)
ax.text(
    (0.5 * (far_x + near_x)).item(),
    (dof_y + torch.tensor(0.02, dtype=torch.float32)).item(),
    "depth of field",
    fontsize=10, color=dark, ha="center", va="bottom",
)

ax.set_xlim(x_min.item(), x_max.item())
ax.set_ylim(y_min.item(), y_max.item())
ax.set_aspect("equal", adjustable="box")
ax.axis("off")

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_10.png", dpi=150, bbox_inches="tight")
plt.show()

The depth of field is set by the lens's focal length, the maximum circle-of-confusion size that still looks sharp, and the camera's *f-number* $N = f / A$, where $A$ is the aperture diameter. The next figure lays out the variables.

The geometry that turns the circle of confusion into a depth-of-field range is two pairs of similar triangles — one for the near limit, one for the far.

**Figure 6.11 — Variables for the depth-of-field calculation.** Two scenes, same focal length, same focused object distance $U$, same circle of confusion $C$ at the sensor — only the aperture differs. The *f-number* $N = f/A$ relates aperture diameter $A$ to focal length: smaller $N$ means a wider aperture. **(Top, $N^a$)** Wider aperture, narrower depth of field — the range $D_1^a$ to $D_2^a$ around $U$ is short. **(Bottom, $N^b$)** Narrower aperture, wider depth of field — same $C$ at the sensor, but a much larger range $D_1^b$ to $D_2^b$ around $U$. (Book §6.3.1, Figure 6.11.)

In [ ]:
# Figure 6.11 — Variables for the depth-of-field calculation (book §6.3.1).
#
# This cell compares two apertures at one focus distance U. The physical knobs
# set f, the focused sensor distance, the near/far depth-of-field limits, and
# the blur circle C; a schematic x-map spaces those quantities like the book.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
n = torch.tensor(1.5, dtype=torch.float32)
R1 = torch.tensor(1.4, dtype=torch.float32)
R2 = torch.tensor(1.4, dtype=torch.float32)
U = torch.tensor(6.0, dtype=torch.float32)
N_a = torch.tensor(3.2, dtype=torch.float32)
N_b = torch.tensor(7.0, dtype=torch.float32)
C = torch.tensor(0.10, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
x_obj = torch.tensor(-1.25, dtype=torch.float32)
x_lens = torch.tensor(2.15, dtype=torch.float32)
x_sensor = torch.tensor(3.95, dtype=torch.float32)
obj_scale = torch.tensor(1.65, dtype=torch.float32)
far_display_factor = torch.tensor(3.25, dtype=torch.float32)
row_y = [torch.tensor(1.85, dtype=torch.float32), torch.tensor(-1.85, dtype=torch.float32)]
lens_gap = torch.tensor(0.0, dtype=torch.float32)
lens_bulge = torch.tensor(0.06, dtype=torch.float32)
lens_half_h = torch.tensor(0.62, dtype=torch.float32)
c_disp = torch.tensor(0.34, dtype=torch.float32)
dof_drop = torch.tensor(0.72, dtype=torch.float32)
tick = torch.tensor(0.08, dtype=torch.float32)

dark, gray = COLORS["dark"], COLORS["gray"]
blue, red = COLORS["blue"], COLORS["red"]
cyan = COLORS["cyan"]

# --- Derived quantities ---
z = torch.tensor(0.0, dtype=torch.float32)
f = focal_length(n, R1, R2)
s = image_distance(U, f)

A_a = f / N_a
A_b = f / N_b
r_a = C / A_a
r_b = C / A_b

v_near_a = s / (1.0 - r_a)
v_far_a = s / (1.0 + r_a)
v_near_b = s / (1.0 - r_b)
v_far_b = s / (1.0 + r_b)

D1_a = f * v_near_a / (v_near_a - f)
D2_a_raw = f * v_far_a / (v_far_a - f)
D1_b = f * v_near_b / (v_near_b - f)
D2_b_raw = f * v_far_b / (v_far_b - f)

D2_a = torch.where(v_far_a > f, D2_a_raw, far_display_factor * U)
D2_b = torch.where(v_far_b > f, D2_b_raw, far_display_factor * U)

img_scale = (x_sensor - x_lens) / s
h_a = torch.tensor(0.85, dtype=torch.float32) * A_a
h_b = torch.tensor(0.85, dtype=torch.float32) * A_b

left_x, right_x, lens_y = lens_curves(lens_gap, lens_bulge, lens_half_h)

def x_obj_map(D):
    D_safe = torch.clamp(D, min=torch.tensor(1.05, dtype=torch.float32) * U)
    return x_obj - obj_scale * torch.log(D_safe / U)

def x_img_map(D):
    return x_lens + img_scale * image_distance(D, f)

def y_at_x(p0, p1, x):
    t = (x - p0[0]) / (p1[0] - p0[0])
    return p0[1] + t * (p1[1] - p0[1])

rows = [
    {"y": row_y[0], "A": A_a, "h": h_a, "D1": D1_a, "D2": D2_a, "tag": "a", "dof_y": row_y[0] - dof_drop},
    {"y": row_y[1], "A": A_b, "h": h_b, "D1": D1_b, "D2": D2_b, "tag": "b", "dof_y": row_y[1] - dof_drop},
]

x_far_a = x_obj_map(D2_a)
x_far_b = x_obj_map(D2_b)

# --- Drawing ---
fig, ax = plt.subplots(figsize=(10.8, 8.0))

ax.plot([x_obj.item(), x_obj.item()], [-3.2, 3.2], color=red, lw=1.0, alpha=0.22, zorder=0)
ax.plot([x_sensor.item(), x_sensor.item()], [-3.2, 3.2], color=cyan, lw=1.0, alpha=0.55, zorder=0)
ax.text(x_obj.item(), 3.12, "object plane", fontsize=14, color=dark, ha="center", va="bottom")
ax.text(x_sensor.item(), 3.12, "sensor plane", fontsize=14, color=dark, ha="center", va="bottom")

for row in rows:
    y0, h = row["y"], row["h"]
    x1, x2 = x_obj_map(row["D1"]), x_obj_map(row["D2"])
    xf1, xf2 = x_img_map(row["D1"]), x_img_map(row["D2"])

    p1 = torch.stack([x1, y0])
    p2 = torch.stack([x2, y0])
    lt = torch.stack([x_lens, y0 + h])
    lb = torch.stack([x_lens, y0 - h])
    f1 = torch.stack([xf1, y0])
    f2 = torch.stack([xf2, y0])
    x_end = x_sensor + torch.tensor(0.28, dtype=torch.float32)

    end_t1 = torch.stack([x_end, y_at_x(lt, f1, x_end)])
    end_b1 = torch.stack([x_end, y_at_x(lb, f1, x_end)])
    end_t2 = torch.stack([x_end, y_at_x(lt, f2, x_end)])
    end_b2 = torch.stack([x_end, y_at_x(lb, f2, x_end)])

    for pA, pB, lw, alpha, zord in [
        (p1, lt, 1.4, 1.0, 2), (p1, lb, 1.4, 1.0, 2),
        (lt, end_t1, 1.4, 1.0, 2), (lb, end_b1, 1.4, 1.0, 2),
        (p2, lt, 0.9, 0.38, 1), (p2, lb, 0.9, 0.38, 1),
        (lt, end_t2, 0.9, 0.38, 1), (lb, end_b2, 0.9, 0.38, 1),
    ]:
        ax.plot([pA[0].item(), pB[0].item()], [pA[1].item(), pB[1].item()], color=blue, lw=lw, alpha=alpha, zorder=zord)

    lens_xy = torch.cat(
        [
            torch.stack([right_x + x_lens, lens_y + y0], dim=1),
            torch.stack([(left_x + x_lens).flip(0), (lens_y + y0).flip(0)], dim=1),
        ],
        dim=0,
    )
    ax.add_patch(
        Polygon([(p[0].item(), p[1].item()) for p in lens_xy], closed=True,
                facecolor="white", edgecolor=dark, lw=1.35, zorder=4)
    )

    xA = x_lens - torch.tensor(0.20, dtype=torch.float32)
    ax.plot([xA.item(), xA.item()], [(y0 - h).item(), (y0 + h).item()], color=dark, lw=1.0, zorder=5)
    ax.plot([(xA - tick).item(), (xA + tick).item()], [(y0 + h).item(), (y0 + h).item()], color=dark, lw=1.0, zorder=5)
    ax.plot([(xA - tick).item(), (xA + tick).item()], [(y0 - h).item(), (y0 - h).item()], color=dark, lw=1.0, zorder=5)
    ax.text((xA - 0.50).item(), y0.item(), rf"$A^{row['tag']}=\frac{{f}}{{N^{row['tag']}}}$",
            fontsize=17, color=dark, ha="center", va="center")

    yC0 = y0 - 0.5 * c_disp
    yC1 = y0 + 0.5 * c_disp
    xC = x_sensor + torch.tensor(0.18, dtype=torch.float32)
    for yy in [yC0, yC1]:
        ax.plot([x_sensor.item(), xC.item()], [yy.item(), yy.item()], color=gray, lw=1.0, ls=":", zorder=3)
    ax.plot([xC.item(), xC.item()], [yC0.item(), yC1.item()], color=gray, lw=1.0, zorder=3)
    ax.plot([(xC - 0.03).item(), (xC + 0.03).item()], [yC0.item(), yC0.item()], color=gray, lw=1.0, zorder=3)
    ax.plot([(xC - 0.03).item(), (xC + 0.03).item()], [yC1.item(), yC1.item()], color=gray, lw=1.0, zorder=3)
    ax.text((xC + 0.08).item(), y0.item(), r"$C$", fontsize=15, color=dark, ha="left", va="center")
    ax.text((xC + 0.42).item(), y0.item(), "circle of\nconfusion", fontsize=11, color=dark, ha="left", va="center")

    yb = row["dof_y"]
    ax.plot([x2.item(), x2.item()], [y0.item(), yb.item()], color=gray, lw=1.0, ls=":", zorder=0)
    ax.plot([x1.item(), x1.item()], [y0.item(), yb.item()], color=gray, lw=1.0, ls=":", zorder=0)
    ax.plot([x2.item(), x1.item()], [yb.item(), yb.item()], color=gray, lw=1.0, ls=":", zorder=0)
    ax.text(((x1 + x2) * 0.5).item(), (yb + 0.07).item(), "depth of field", fontsize=14, color=dark, ha="center", va="bottom")

    ax.text((x2 + 0.02).item(), (y0 - 0.10).item(), rf"$D_2^{row['tag']}$", fontsize=18, color=dark, ha="left", va="center")
    ax.text((x1 - 0.02).item(), (y0 - 0.10).item(), rf"$D_1^{row['tag']}$", fontsize=18, color=dark, ha="right", va="center")

ax.plot([x_obj.item(), x_lens.item()], [0.55, 0.55], color=gray, lw=1.0, ls=":", zorder=0)
ax.text(((x_obj + x_lens) * 0.5).item(), 0.63, r"$U$", fontsize=20, color=dark, ha="center", va="bottom")

x_cu = x_obj_map(D2_a) + torch.tensor(0.52, dtype=torch.float32)
y_cu = row_y[0] + torch.tensor(0.22, dtype=torch.float32)
ax.text(x_cu.item(), y_cu.item(), r"$\approx \frac{CU}{f}$", fontsize=15, color=gray, ha="center", va="center")

y_arrow = -0.35
ax.add_patch(
    FancyArrowPatch(
        (x_far_a.item(), y_arrow),
        (x_far_b.item(), y_arrow),
        arrowstyle="-|>",
        mutation_scale=14,
        linewidth=1.4,
        color=red,
        zorder=6,
    )
)
ax.text(((x_far_a + x_far_b) * 0.5).item(), y_arrow + 0.10, "larger depth of field",
        fontsize=12, color=red, ha="center", va="bottom")

xN = x_lens + torch.tensor(0.03, dtype=torch.float32)
yN = row_y[1]
ax.add_patch(FancyArrowPatch((xN.item(), (yN + h_a).item()), (xN.item(), (yN + h_b).item()),
                             arrowstyle="-|>", mutation_scale=12, color=red, lw=1.2, zorder=6))
ax.add_patch(FancyArrowPatch((xN.item(), (yN - h_a).item()), (xN.item(), (yN - h_b).item()),
                             arrowstyle="-|>", mutation_scale=12, color=red, lw=1.2, zorder=6))
ax.text((xN + 0.12).item(), (yN - 0.18).item(), "narrowed\naperture", fontsize=12, color=red, ha="left", va="center")

ax.set_xlim((x_far_b - 0.25).item(), (x_sensor + 1.15).item())
ax.set_ylim(-3.35, 3.45)
ax.axis("off")

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_11.png", dpi=150, bbox_inches="tight")
plt.show()

The two similar-triangle pairs in Figure 6.11 yield expressions for $D_1$ and $D_2$ that, when solved and summed, give the exact depth of field:

$$D = \frac{2 N C U^2 f^2}{f^4 - N^2 C^2 U^2}.$$

For the practical regime $C \ll f/N$, the second term in the denominator drops out and the formula collapses to a clean proportionality:

$$D \approx \frac{2 N C U^2}{f^2}.$$

Depth of field is *linear* in $N$. Doubling $N$ doubles the in-focus range — but the light hitting the sensor falls as $1/N^2$, so the same scene needs four times the exposure. The next figure shows the trade-off on a real (synthetic) ruler.

**Figure 6.12 — Photographic depth of field as a function of aperture.** One sharp photograph of a ruler, blurred at each pixel by an amount proportional to its depth offset from the focal plane and scaled by $1/N$ — recreating the book's Figure 6.12 photographic demonstration. **(a) $f/2$:** narrow sharp band, blurred ends. **(b) $f/4$:** doubled DOF. **(c) $f/8$:** nearly the whole ruler sharp. (Book §6.3.1, Figure 6.12. Recreated by applying a Gaussian blur with $\sigma \propto 1/N$; the $f/2$ reference blur is calibrated visually.)

In [ ]:
# Figure 6.12 — Photographic depth of field as a function of aperture (book §6.3.1).
#
# This cell starts from one sharp ruler image and adds depth-dependent defocus.
# The blur radius is proportional to aperture diameter (and inversely to the
# f-number N), so a larger f-number narrows the blur disk and widens the
# region of acceptable sharpness — recreating the photographic effect shown
# in the book's Figure 6.12.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. The f/2 reference blur is calibrated visually; f/4 and
# f/8 follow from sigma ∝ aperture diameter ∝ 1/N.
f_numbers = torch.tensor([2.0, 4.0, 8.0], dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
asset_path = ASSETS_DIR / "ruler_single.png"
focus_row_frac = torch.tensor(0.54, dtype=torch.float32)
sharp_band_frac = torch.tensor(0.06, dtype=torch.float32)
sigma_f2_px = torch.tensor(8.0, dtype=torch.float32)
n_blur_levels = 9
panel_labels = ["(a) f/2.0", "(b) f/4.0", "(c) f/8.0"]

# --- Derived quantities ---
img = torch.tensor(plt.imread(asset_path), dtype=torch.float32)
if img.ndim != 3:
    raise ValueError("The ruler asset must be an RGB image.")
if img.shape[-1] == 4:
    img = img[..., :3]
if img.max() > 1.5:
    img = img / 255.0

H, W, _ = img.shape
img_bchw = img.permute(2, 0, 1).unsqueeze(0).to(DEVICE)

# Per-row blur strength in [0, 1]: zero inside the sharp band around the focal
# row, ramping up to one at the edge of the image. Broadcast to (H, W) so every
# column at a given row gets the same blur level (depth varies with image row
# for this slanted ruler scene).
focus_row = focus_row_frac * torch.tensor(H - 1, dtype=torch.float32)
far_row = torch.maximum(focus_row, torch.tensor(H - 1, dtype=torch.float32) - focus_row)
row_y = torch.arange(H, dtype=torch.float32)
row_dist = torch.abs(row_y - focus_row) / far_row
blur_strength = torch.clamp(
    (row_dist - sharp_band_frac) / torch.clamp(1.0 - sharp_band_frac, min=1e-6),
    min=0.0,
    max=1.0,
)
blur_strength_2d = blur_strength[:, None].expand(H, W)

# --- Blur each panel ---
# For each f-number, precompute n_blur_levels blurred versions of the source
# image at sigmas linearly spaced up to sigma_max, then assign each pixel the
# blur level closest to its row's blur strength.
panels = []
for N in f_numbers:
    sigma_max = sigma_f2_px * f_numbers[0] / N
    sigmas = torch.linspace(0.0, sigma_max.item(), n_blur_levels)
    bank = []
    for s in sigmas:
        if s.item() <= 1e-6:
            bank.append(img_bchw)
        else:
            kernel_size = int(2 * torch.ceil(3.0 * s).item() + 1)
            bank.append(kornia.filters.gaussian_blur2d(
                img_bchw, (kernel_size, kernel_size), (s.item(), s.item())
            ))
    bank = torch.stack([b.squeeze(0).permute(1, 2, 0) for b in bank], dim=0)

    idx = torch.round(blur_strength_2d * (n_blur_levels - 1)).to(torch.long)
    panel = torch.empty_like(img)
    for k in range(n_blur_levels):
        mask = idx == k
        panel[mask] = bank[k].to(panel.device)[mask]
    panels.append(torch.clamp(panel, 0.0, 1.0))

# --- Drawing ---
fig, axes = plt.subplots(1, 3, figsize=(9.8, 6.8))
for ax, panel, label in zip(axes, panels, panel_labels):
    ax.imshow(panel)
    ax.axis("off")
    ax.text(
        0.5, -0.035, label,
        transform=ax.transAxes,
        fontsize=13,
        color=COLORS["dark"],
        ha="center",
        va="top",
    )
fig.subplots_adjust(left=0.02, right=0.98, top=0.99, bottom=0.08, wspace=0.08)

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_12.png", dpi=150, bbox_inches="tight")
plt.show()

The photographic trade-off — wider aperture, more light but shallower DOF — is what the formula $D \approx 2NCU^2/f^2$ makes quantitative. §6.3.2 turns the lens curvature inside out and asks what changes with a *concave* lens.

### 6.3.2 — Concave Lenses

The lens designed in §6.2 was *convex* — both surfaces bowing outward, focal length positive. Reverse the curvature on both surfaces and you get a *concave* lens, with both surfaces bowing inward. In the lensmaker's formula

$$\frac{1}{f} = (n-1)\left(\frac{1}{R_1} + \frac{1}{R_2}\right),$$

a concave surface contributes the same magnitude with the opposite sign, so $f$ comes out *negative*. The same paraxial geometry that focuses rays to a real point past a convex lens now bends them *away* from the axis at a concave lens — but the back-projection of those diverging rays still meets at a single point, on the *source side* of the lens. That point is the lens's *virtual* focal point.

**Figure 6.13 — Convex and concave thin-lens behavior.** **(a)** A convex lens with focal length $+f$: parallel rays from the left converge to a *real* focal point at distance $f$ past the lens. **(b)** A concave lens with focal length $-f$: parallel rays diverge after the lens, but their back-projections (dotted) meet at a *virtual* focal point at distance $f$ on the source side. **(c)** Same concave lens with a tilted parallel bundle: the virtual focal point shifts off-axis (cyan), just as the focal point in a convex lens shifts to image off-axis sources — the lensmaker's formula handles both cases with one sign change. (Book §6.3.2, Figure 6.13.)

In [ ]:
# Figure 6.13 (a–c) — Convex and concave thin-lens behavior (book §6.3.2).
#
# This cell draws three panels: a converging lens focusing parallel rays to a
# real point, a diverging lens forming a virtual focus on the source side under
# parallel input, and the same diverging lens under a tilted parallel bundle.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
f_conv = torch.tensor(0.82, dtype=torch.float32)
f_div = torch.tensor(0.68, dtype=torch.float32)
c = torch.tensor(0.34, dtype=torch.float32)
tilt_c = torch.tensor(-0.18, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
lens_half_h = torch.tensor(0.60, dtype=torch.float32)
convex_bulge = torch.tensor(0.11, dtype=torch.float32)
concave_waist = torch.tensor(0.035, dtype=torch.float32)
concave_edge = torch.tensor(0.055, dtype=torch.float32)
x_left = torch.tensor(-1.45, dtype=torch.float32)
x_right = torch.tensor(1.25, dtype=torch.float32)
dark, blue, cyan, panel_gray = COLORS["dark"], COLORS["blue"], COLORS["cyan"], COLORS["panel"]

# --- Derived quantities ---
z = torch.tensor(0.0, dtype=torch.float32)
y_rays = torch.tensor([c.item(), 0.0, -c.item()], dtype=torch.float32)

left_x, right_x, lens_y = lens_curves(z, convex_bulge, lens_half_h)
convex_xy = torch.cat(
    [torch.stack([right_x, lens_y], dim=1), torch.stack([left_x.flip(0), lens_y.flip(0)], dim=1)],
    dim=0,
)

y_conc = torch.linspace(-lens_half_h, lens_half_h, 240, dtype=torch.float32)
s_conc = torch.sqrt(torch.clamp(1.0 - (y_conc / lens_half_h) ** 2, min=0.0))
x_conc = concave_waist + concave_edge * (1.0 - s_conc)
concave_xy = torch.cat(
    [torch.stack([x_conc, y_conc], dim=1), torch.stack([-x_conc.flip(0), y_conc.flip(0)], dim=1)],
    dim=0,
)

x_focus_a = f_conv
x_focus_b = -f_div
y_focus_c = -tilt_c * f_div

# --- Drawing ---
fig, axes = plt.subplots(1, 3, figsize=(9.2, 3.0))

for ax in axes:
    ax.set_xlim(x_left.item(), x_right.item())
    ax.set_ylim(-0.82, 0.82)
    ax.axis("off")

# (a) Convex lens — parallel rays focus to a real point.
ax = axes[0]
ax.add_patch(
    Polygon([(p[0].item(), p[1].item()) for p in convex_xy], closed=True,
            facecolor="white", edgecolor=dark, lw=1.2, zorder=3)
)
ax.scatter([x_focus_a.item()], [0.0], s=10, color=blue, zorder=5)
for y in y_rays:
    ax.plot([x_left.item(), 0.0], [y.item(), y.item()], color=blue, lw=1.15, zorder=2)
for y in [c, -c]:
    ax.plot([0.0, x_focus_a.item()], [y.item(), 0.0], color=blue, lw=1.15, zorder=2)
ax.plot([0.0, x_right.item()], [0.0, 0.0], color=blue, lw=1.15, zorder=2)
ax.text(0.0, -0.73, "(a)", fontsize=13, color=panel_gray, style="italic", ha="center", va="center")

# (b) Concave lens — parallel rays diverge; back-projection meets at virtual focus.
ax = axes[1]
ax.add_patch(
    Polygon([(p[0].item(), p[1].item()) for p in concave_xy], closed=True,
            facecolor="white", edgecolor=dark, lw=1.2, zorder=3)
)
ax.scatter([x_focus_b.item()], [0.0], s=10, color=blue, zorder=5)
for y in y_rays:
    ax.plot([x_left.item(), 0.0], [y.item(), y.item()], color=blue, lw=1.15, zorder=2)
    if torch.abs(y).item() > 1e-6:
        ax.plot([x_focus_b.item(), 0.0], [0.0, y.item()], color=blue, lw=1.0, ls=(0, (2.2, 2.2)), zorder=1)
        m = y / f_div
        ax.plot([0.0, x_right.item()], [y.item(), (y + m * x_right).item()], color=blue, lw=1.15, zorder=2)
    else:
        ax.plot([0.0, x_right.item()], [0.0, 0.0], color=blue, lw=1.15, zorder=2)
ax.text(0.0, -0.73, "(b)", fontsize=13, color=panel_gray, style="italic", ha="center", va="center")

# (c) Concave lens with tilted incident bundle — virtual focus shifts off-axis.
ax = axes[2]
ax.add_patch(
    Polygon([(p[0].item(), p[1].item()) for p in concave_xy], closed=True,
            facecolor="white", edgecolor=dark, lw=1.2, zorder=3)
)
ax.scatter([x_focus_b.item()], [y_focus_c.item()], s=10, color=cyan, zorder=5)

# Horizontal axis underneath, separate from the tilted bundle.
ax.plot([x_left.item(), x_right.item()], [0.0, 0.0], color=blue, lw=1.15, zorder=1)

# Three tilted incident rays, back-projected virtual rays, and refracted exit rays.
for y in [c, z, -c]:
    yL = y + tilt_c * x_left
    ax.plot([x_left.item(), 0.0], [yL.item(), y.item()], color=cyan, lw=1.15, zorder=2)
for y in [c, -c]:
    ax.plot([x_focus_b.item(), 0.0], [y_focus_c.item(), y.item()], color=cyan, lw=1.0, ls=(0, (2.2, 2.2)), zorder=1)
for y in [c, z, -c]:
    m = tilt_c + y / f_div
    ax.plot([0.0, x_right.item()], [y.item(), (y + m * x_right).item()], color=cyan, lw=1.15, zorder=2)

ax.text(0.0, -0.73, "(c)", fontsize=13, color=panel_gray, style="italic", ha="center", va="center")

fig.subplots_adjust(left=0.03, right=0.99, top=0.98, bottom=0.08, wspace=0.13)

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_13.png", dpi=150, bbox_inches="tight")
plt.show()

A concave lens alone can't form a real image — it diverges every bundle it sees. But paired with a convex lens, that diverging behavior becomes useful: if the concave lens sits at the convex lens's focal point, the two together turn a converging bundle back into parallel rays — at a *different* angle than they entered. That angular amplification is the principle behind the Galilean telescope, which §6.3.3 builds next.

### 6.3.3 — Lenses in a Telescope

A convex lens and a concave lens together form a *Galilean telescope*, named after the one Galileo built in 1609. The construction is simple: place a convex lens (lens 1, focal length $f_1$, long) with a concave lens (lens 2, focal length $f_2$, shorter) such that they share a common focal point — lens 1 is distance $f_1$ to the left of that shared point, lens 2 is distance $f_2$ to the same side. The lenses sit $f_1 - f_2$ apart. In that configuration, parallel rays entering lens 1 converge toward the shared focal point, and lens 2 — intercepting them before they reach it — refracts them back into a parallel bundle. The output bundle is parallel like the input, but compressed into a smaller cross-section.

What makes it a *telescope* is what happens when the input direction tilts.

**Figure 6.14 — Galilean telescope geometry.**

**(a) Parallel input, parallel output.** Three parallel rays enter lens 1 from the left, converge toward the shared focal point, and are intercepted by lens 2 before reaching it — refracting back into a parallel bundle on the right. The output rays are parallel like the input, but closer together, compressed into the smaller exit aperture.

**(b) Tilted input, angular magnification.** When the input bundle tilts at angle $\delta_i$ from the optical axis, the chief ray through lens 1's center reaches the shared focal point at height $d = f_1 \delta_i$ above the axis (small-angle approximation, property 3 of §6.3). The same point $d$ is at distance $f_2$ from lens 2, which refracts the bundle into a parallel output at angle $\delta_o = d / f_2$ from the axis. The two similar triangles share the height $d$ but have different base lengths $f_1$ and $f_2$, giving the magnification $M = \delta_o / \delta_i = f_1 / f_2$. (Book §6.3.3, Figure 6.14.)

In [ ]:
# Figure 6.14 (a, b) — Galilean telescope (book §6.3.3).
#
# (a) Parallel rays enter the convex objective lens 1, converge toward the
# shared focal point, and the concave eyepiece lens 2 — placed f_2 before that
# point — sends them out as a parallel bundle. The output bundle is compressed.
#
# (b) A tilted input bundle hits lens 1 at angle δ_i. The chief ray through
# lens 1's center reaches the shared focal point at height d = f_1 · δ_i.
# Lens 2 sends the bundle out at the larger angle δ_o = d/f_2, giving the
# magnification M = δ_o/δ_i = f_1/f_2.

# --- Tunable physics parameters ---
# The parameters below drive the figure's geometry within a narrow range
# around the defaults. Significant deviations may push labels off-figure or
# break the lens-shape proportions, because some visual layout constants are
# not yet coupled to the physics. The math itself remains correct for any inputs.
f1 = torch.tensor(3.9, dtype=torch.float32)
f2 = torch.tensor(1.35, dtype=torch.float32)
ray_h = torch.tensor(0.62, dtype=torch.float32)
delta_i = torch.deg2rad(torch.tensor(5.5, dtype=torch.float32))
y1_hi = torch.tensor(0.95, dtype=torch.float32)
y1_lo = torch.tensor(0.03, dtype=torch.float32)

# --- Visual layout knobs (cosmetic, no physics meaning) ---
x_left = torch.tensor(-2.9, dtype=torch.float32)
x_right_pad = torch.tensor(1.35, dtype=torch.float32)
lens1_h = torch.tensor(1.65, dtype=torch.float32)
lens2_h = torch.tensor(0.92, dtype=torch.float32)
lens1_t = torch.tensor(0.14, dtype=torch.float32)
lens2_t = torch.tensor(0.18, dtype=torch.float32)
ref_len = torch.tensor(0.42, dtype=torch.float32)
arc_r_i = torch.tensor(0.34, dtype=torch.float32)
arc_r_o = torch.tensor(0.42, dtype=torch.float32)
tick = torch.tensor(0.07, dtype=torch.float32)
dash = (0, (2, 2))

dark, blue, panel_gray = COLORS["dark"], COLORS["blue"], COLORS["panel"]
violet = "#c06df2"  # ray color for panel (b); consolidation pass may move to COLORS

# --- Derived quantities ---
z = torch.tensor(0.0, dtype=torch.float32)
x1 = z
x2 = f1 - f2
xF = f1
x_right = xF + x_right_pad

# Lens 1: thin biconvex.
u = torch.linspace(-1.0, 1.0, 240, dtype=torch.float32)
lens1_profile = lens1_t * torch.sqrt(torch.clamp(1.0 - u**2, min=0.0))
lens1_xy = torch.cat([
    torch.stack([x1 + lens1_profile, lens1_h * u], dim=1),
    torch.stack([(x1 - lens1_profile).flip(0), (lens1_h * u).flip(0)], dim=1),
], dim=0)

# Lens 2: biconcave (waist narrower than edges).
lens2_profile = lens2_t * (0.30 + 0.70 * (1.0 - torch.sqrt(torch.clamp(1.0 - u**2, min=0.0))))
lens2_xy = torch.cat([
    torch.stack([x2 + lens2_profile, lens2_h * u], dim=1),
    torch.stack([(x2 - lens2_profile).flip(0), (lens2_h * u).flip(0)], dim=1),
], dim=0)


def lens_polygon(xy):
    return Polygon(xy.tolist(), closed=True, facecolor="white", edgecolor=dark, lw=1.4, zorder=4)


# --- Drawing ---
fig, axes = plt.subplots(2, 1, figsize=(11.5, 8.5))

# === Panel (a): parallel in, parallel out ===
ax = axes[0]
rays = torch.stack([ray_h, z, -ray_h])
rays_at_lens2 = rays * (f2 / f1)

ax.plot([x_left.item(), x_right.item()], [0.0, 0.0], color=blue, lw=0.9, alpha=0.55, zorder=0)
ax.add_patch(lens_polygon(lens1_xy))
ax.add_patch(lens_polygon(lens2_xy))

for y0, y2v in zip(rays, rays_at_lens2):
    ax.plot([x_left.item(), x1.item()], [y0.item(), y0.item()], color=blue, lw=1.1, zorder=3)
    ax.plot([x1.item(), x2.item()], [y0.item(), y2v.item()], color=blue, lw=1.1, zorder=3)
    ax.plot([x2.item(), x_right.item()], [y2v.item(), y2v.item()], color=blue, lw=1.1, zorder=3)
    ax.plot([x2.item(), xF.item()], [y2v.item(), 0.0], color=blue, lw=1.0, ls=dash, zorder=2)

ax.scatter([xF.item()], [0.0], s=38, color=blue, zorder=5)

br_y_a = torch.tensor(-1.55, dtype=torch.float32)
for xa, xb, yb, lab in [
    (x1, xF, br_y_a, r"$f_1$"),
    (x2, xF, br_y_a + torch.tensor(0.30, dtype=torch.float32), r"$f_2$"),
]:
    ax.plot([xa.item(), xb.item()], [yb.item(), yb.item()], color=dark, lw=1.0)
    for x_end in [xa, xb]:
        ax.plot([x_end.item(), x_end.item()], [(yb - tick).item(), (yb + tick).item()], color=dark, lw=1.0)
    ax.text((0.5 * (xa + xb)).item(), (yb + 0.12).item(), lab, fontsize=16, color=dark, ha="center", va="bottom")

ax.text((x1 + 0.20).item(), 1.75, "lens 1", fontsize=15, color=blue, ha="center", va="center")
ax.text((x2 + 0.20).item(), 1.20, "lens 2", fontsize=15, color=blue, ha="center", va="center")
ax.text((0.5 * (x_left + x_right)).item(), -2.10, "(a)", fontsize=17, color=panel_gray, style="italic", ha="center", va="center")

ax.set_xlim((x_left - 0.25).item(), (x_right + 0.25).item())
ax.set_ylim(-2.25, 2.05)
ax.axis("off")

# === Panel (b): tilted input, angular magnification ===
ax = axes[1]

m_in = -torch.tan(delta_i)
yF = y1_lo + m_in * (xF - x1)
d = -yF
delta_o = torch.atan(d / f2)
m_out = -torch.tan(delta_o)


def ray_bundle(y1):
    yL = y1 + m_in * (x_left - x1)
    m_mid = (yF - y1) / (xF - x1)
    y2 = y1 + m_mid * (x2 - x1)
    yR = y2 + m_out * (x_right - x2)
    return (
        torch.stack([x_left, yL]),
        torch.stack([x1, y1]),
        torch.stack([x2, y2]),
        torch.stack([x_right, yR]),
    )


ray_hi = ray_bundle(y1_hi)
ray_lo = ray_bundle(y1_lo)
pF = torch.stack([xF, yF])
pD = torch.stack([xF, z])

ax.plot([x_left.item(), x_right.item()], [0.0, 0.0], color=blue, lw=1.0, alpha=0.45, zorder=1)
ax.add_patch(lens_polygon(lens1_xy))
ax.add_patch(lens_polygon(lens2_xy))

for ray, lw, alpha in [(ray_hi, 2.0, 1.0), (ray_lo, 1.0, 0.7)]:
    pL, p1, p2, pR = ray
    for pA, pB in [(pL, p1), (p1, p2), (p2, pR)]:
        ax.plot([pA[0].item(), pB[0].item()], [pA[1].item(), pB[1].item()], color=violet, lw=lw, alpha=alpha, zorder=3)

# Back-projection to the shared focal point.
for p2 in [ray_hi[2], ray_lo[2]]:
    ax.plot([p2[0].item(), pF[0].item()], [p2[1].item(), pF[1].item()], color=violet, lw=1.0, ls=dash, alpha=0.85, zorder=2)

# Focal-length brackets.
br_y_b = torch.tensor(-1.55, dtype=torch.float32)
for xa, xb, yb, lab in [
    (x1, xF, br_y_b, r"$f_1$"),
    (x2, xF, br_y_b + torch.tensor(0.30, dtype=torch.float32), r"$f_2$"),
]:
    ax.plot([xa.item(), xb.item()], [yb.item(), yb.item()], color=dark, lw=1.0)
    for x_end in [xa, xb]:
        ax.plot([x_end.item(), x_end.item()], [(yb - tick).item(), (yb + tick).item()], color=dark, lw=1.0)
    ax.text((0.5 * (xa + xb)).item(), (yb + 0.12).item(), lab, fontsize=16, color=dark, ha="center", va="bottom")

# d drop at the shared focal plane (lower-right corner of the d label, near the focal point dot).
ax.scatter([pD[0].item()], [pD[1].item()], s=44, color=blue, zorder=5)
ax.annotate("", xy=(pD[0].item(), pF[1].item()), xytext=(pD[0].item(), pD[1].item()),
            arrowprops=dict(arrowstyle="->", lw=1.0, color=dark))
ax.text((pD[0] + 0.07).item(), (0.5 * (pD[1] + pF[1])).item(), r"$d$", fontsize=16, color=dark, ha="left", va="center")

# δ_i labels — one on each incoming ray (parallel input bundle, both at same angle).
t = torch.linspace(0.0, 1.0, 60, dtype=torch.float32)
arc_i = -delta_i + t * delta_i
arc_o = -delta_o + t * delta_o

for ray in [ray_hi, ray_lo]:
    y_ref = ray[0][1] - torch.tensor(0.10, dtype=torch.float32)
    # Horizontal reference line on the left of each input ray.
    ax.plot([(x_left + 0.04).item(), (x_left + ref_len).item()], [y_ref.item(), y_ref.item()], color=dark, lw=1.0)
    # Angle arc opening between the ray and the horizontal reference.
    c_i = torch.stack([x_left + 0.32, y_ref])
    ax.plot((c_i[0] + arc_r_i * torch.cos(arc_i)).tolist(),
            (c_i[1] + arc_r_i * torch.sin(arc_i)).tolist(),
            color=dark, lw=1.0)
    # δ_i label to the left of each arc.
    ax.text((x_left + 0.12).item(), (ray[0][1] - 0.30).item(), r"$\delta_i$",
            fontsize=18, color=dark, ha="right", va="center")

# δ_o — single label on the lower exit ray near the focal point.
# Anchor the arc just to the right of the d marker, at the axis (where the lower exit ray crosses).
c_o = torch.stack([xF + torch.tensor(0.15, dtype=torch.float32), torch.tensor(-0.02, dtype=torch.float32)])
# Horizontal reference line at the axis, extending right from the d point.
ax.plot([c_o[0].item(), (c_o[0] + arc_r_o).item()], [c_o[1].item(), c_o[1].item()], color=dark, lw=1.0)
# Angle arc opening downward between the horizontal reference and the lower exit ray.
ax.plot((c_o[0] + arc_r_o * torch.cos(arc_o)).tolist(),
        (c_o[1] + arc_r_o * torch.sin(arc_o)).tolist(),
        color=dark, lw=1.0)
# δ_o label to the right of the arc.
ax.text((c_o[0] + arc_r_o + 0.05).item(), (c_o[1] - 0.18).item(), r"$\delta_o$",
        fontsize=18, color=dark, ha="left", va="center")

ax.text((x1 + 0.20).item(), 1.75, "lens 1", fontsize=15, color=blue, ha="center", va="center")
ax.text((x2 + 0.20).item(), 1.20, "lens 2", fontsize=15, color=blue, ha="center", va="center")
ax.text((0.5 * (x_left + x_right)).item(), -2.10, "(b)", fontsize=17, color=panel_gray, style="italic", ha="center", va="center")

ax.set_xlim((x_left - 0.25).item(), (x_right + 0.25).item())
ax.set_ylim(-2.25, 2.05)
ax.axis("off")

fig.subplots_adjust(left=0.04, right=0.98, top=0.98, bottom=0.04, hspace=0.18)

# --- Save ---
plt.savefig(IMAGES_DIR / "fig06_14ab.png", dpi=150, bbox_inches="tight")
plt.show()

With $f_1 > f_2$ the telescope magnifies. Galileo built his at $M \approx 30$ by pairing a long convex objective with a short concave eyepiece; the book's Figs 6.15 and 6.16 show a cardboard recreation ($M \approx 27$ from $f_1 = 500$ mm, $f_2 = 18$ mm) and the moon through it, alongside Galileo's own lunar drawings. Those photographs aren't reproduced here.

With this, §6.3 is complete. The chapter has worked from Snell's law through the lensmaker's formula and used it for imaging, depth of field, concave lenses, and the telescope.